In [60]:
from __future__ import annotations 

import hashlib
import json 
import logging
import os
import random
import re
import sys
import time 
import warnings
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, Generator, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch 

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

In [61]:
def setup_logger(name: str = "hallucination_engine", log_dir: Path = Path("logs")) -> logging.Logger:
    log_dir.mkdir(parents=True, exist_ok=True)
    log_path = log_dir/f"{name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

    fmt = logging.Formatter(
        fmt="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
        datefmt = "%Y-%m-%d %H:%M:%S",
    )

    logger = logging.getLogger(name)
    logger.setLevel(logging.DEBUG)

    # avoid duplicate handlers
    if logger.handlers:
        logger.handlers.clear()

    ch = logging.StreamHandler(sys.stdout)
    ch.setLevel(logging.INFO)
    ch.setFormatter(fmt)

    fh = logging.FileHandler(log_path, encoding= "utf-8")
    fh.setLevel(logging.DEBUG)
    fh.setFormatter(fmt)

    logger.addHandler(ch)
    logger.addHandler(fh)

    return logger 

log = setup_logger()
log.info("Logger initialised - Python %s", sys.version.split()[0])

2026-07-02 19:02:04 | INFO     | hallucination_engine | Logger initialised - Python 3.10.20


In [62]:
SEED: int = 42

def set_seed(seed: int = SEED) -> None:
    """Set random seeds for Python, Numpy and PyTorch"""

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASSEED"] = str(seed)

set_seed(SEED)

DEVICE: torch.device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

log.info("Active device: %s", DEVICE)

if DEVICE.type == "cuda":
    log.info(
        "GPU: %s | VRAM: %.1f GB",
        torch.cuda.get_device_name(0),
        torch.cuda.get_device_properties(0).total_memory / 1e9,
    )


2026-07-02 19:02:04 | INFO     | hallucination_engine | Active device: cuda
2026-07-02 19:02:04 | INFO     | hallucination_engine | GPU: NVIDIA GeForce RTX 4070 Laptop GPU | VRAM: 8.6 GB


In [63]:
from dataclasses import dataclass, field, asdict

@dataclass
class PipelineConfig:
    """Central configuration for the hallucination detection pipeline.

    Every path is relative to the project root.  Downstream sections should
    read paths exclusively from this object so that moving the project root
    only requires changing `PROJECT_ROOT`.
    """

    PROJECT_ROOT: Path = field(default_factory=lambda: Path(".").resolve())

    # Core data directories
    DATA_DIR: Path = field(init=False)
    PDF_DIR: Path = field(init=False)
    PARSED_DIR: Path = field(init=False)
    CHUNKS_DIR: Path = field(init=False)

    EMBEDDINGS_DIR: Path = field(init=False)
    VECTOR_DB_PATH: Path = field(init=False)
    ANNOTATIONS_DIR: Path = field(init=False)
    MODEL_OUTPUT_DIR: Path = field(init=False)
    OUTPUTS_DIR: Path = field(init=False)
    LOGS_DIR: Path = field(init=False)

    # Database
    ANNOTATION_DB: str = "sqlite:///annotations/annotations.db"

    # Chunking 
    CHUNK_SIZE: int = 512          # tokens per chunk
    CHUNK_OVERLAP: int = 64        # overlap between consecutive chunks

    # Models
    EMBEDDING_MODEL: str = "sentence-transformers/all-mpnet-base-v2"
    NLI_MODEL: str = "cross-encoder/nli-deberta-v3-small"
    RERANKER_MODEL: str = "cross-encoder/ms-marco-MiniLM-L-6-v2"
    CLAIM_EXTRACTOR_MODEL: str = "facebook/bart-large-cnn"

    # Retrieval 
    TOP_K: int = 5                 # evidence chunks to retrieve per claim
    VECTOR_BACKEND: str = "faiss"  # "faiss" | "chroma"

    # Training
    BATCH_SIZE: int = 8
    GRADIENT_ACCUMULATION_STEPS: int = 4   # effective batch = 32
    LEARNING_RATE: float = 1e-5
    EPOCHS: int = 5
    WARMUP_RATIO: float = 0.06
    WEIGHT_DECAY: float = 0.01
    FP16: bool = torch.cuda.is_available()  # mixed precision on CUDA
    MAX_SEQ_LENGTH: int = 512

    # Dataset splits
    TRAIN_RATIO: float = 0.70
    VAL_RATIO: float = 0.15
    TEST_RATIO: float = 0.15

    # SEC downloader
    # Identity string required by SEC EDGAR fair-use policy
    SEC_USER_AGENT: str = "TRResearch research@thomsonreuters.com"
    SUPPORTED_FORMS: Tuple[str, ...] = ("10-K", "10-Q", "8-K")
    DEFAULT_AFTER_DATE: str = "2022-01-01"
    DEFAULT_BEFORE_DATE: str = "2024-12-31"
    MAX_FILINGS_PER_TICKER: int = 3  

    def __post_init__(self)-> None:

        root = self.PROJECT_ROOT
        self.DATA_DIR = root / "data"
        self.PDF_DIR = self.DATA_DIR / "raw_documents"
        self.PARSED_DIR = self.DATA_DIR / "parsed"
        self.CHUNKS_DIR = self.DATA_DIR / "chunks"
        self.EMBEDDINGS_DIR = self.DATA_DIR / "embeddings"
        self.VECTOR_DB_PATH = self.DATA_DIR / "vector_db"
        self.ANNOTATIONS_DIR = self.DATA_DIR / "annotations"
        self.MODEL_OUTPUT_DIR = root / "models" / "fine_tuned_model"
        self.OUTPUTS_DIR = root / "outputs"
        self.LOGS_DIR = root / "logs"
        self._create_directories()

    def _create_directories(self)-> None:
        """Idempotently create the full project directory tree."""
        dirs = [
            self.DATA_DIR, self.PDF_DIR, self.PARSED_DIR,
            self.CHUNKS_DIR, self.EMBEDDINGS_DIR, self.VECTOR_DB_PATH,
            self.ANNOTATIONS_DIR, self.MODEL_OUTPUT_DIR,
            self.OUTPUTS_DIR, self.LOGS_DIR,
        ]
        for d in dirs:
            d.mkdir(parents=True, exist_ok=True)

    def to_json(self, path: Optional[Path] = None) -> str:
        """Serialise config to JSON (for reproducibility logs)."""
        d = {k: str(v) for k, v in asdict(self).items()}
        j = json.dumps(d, indent=2)
        if path:
            path.write_text(j, encoding="utf-8")
        return j

    @classmethod
    def from_json(cls, path: Path) -> "PipelineConfig":
        """Load config from a previously saved JSON file."""
        raw = json.loads(path.read_text(encoding="utf-8"))
        # Only pass fields that exist in the dataclass __init__
        init_fields = {"PROJECT_ROOT", "ANNOTATION_DB", "CHUNK_SIZE",
                       "CHUNK_OVERLAP", "EMBEDDING_MODEL", "NLI_MODEL",
                       "RERANKER_MODEL", "CLAIM_EXTRACTOR_MODEL", "TOP_K",
                       "VECTOR_BACKEND", "BATCH_SIZE",
                       "GRADIENT_ACCUMULATION_STEPS", "LEARNING_RATE",
                       "EPOCHS", "WARMUP_RATIO", "WEIGHT_DECAY", "FP16",
                       "MAX_SEQ_LENGTH", "TRAIN_RATIO", "VAL_RATIO",
                       "TEST_RATIO", "SEC_USER_AGENT", "SUPPORTED_FORMS",
                       "DEFAULT_AFTER_DATE", "DEFAULT_BEFORE_DATE",
                       "MAX_FILINGS_PER_TICKER"}
        kwargs = {k: v for k, v in raw.items() if k in init_fields}
        if "PROJECT_ROOT" in kwargs:
            kwargs["PROJECT_ROOT"] = Path(kwargs["PROJECT_ROOT"])
        return cls(**kwargs)


# Instantiate global config
cfg = PipelineConfig()

# Persist config for reproducibility
cfg_path = cfg.OUTPUTS_DIR / "pipeline_config.json"
cfg.to_json(cfg_path)

log.info("Configuration initialised — project root: %s", cfg.PROJECT_ROOT)
log.info("Config saved to %s", cfg_path)
log.info("Vector backend: %s  |  NLI model: %s", cfg.VECTOR_BACKEND, cfg.NLI_MODEL)
    

2026-07-02 19:02:04 | INFO     | hallucination_engine | Configuration initialised — project root: /home/lonewolf/quant/Projects/FINANCIAL_LLM/v3
2026-07-02 19:02:04 | INFO     | hallucination_engine | Config saved to /home/lonewolf/quant/Projects/FINANCIAL_LLM/v3/outputs/pipeline_config.json
2026-07-02 19:02:04 | INFO     | hallucination_engine | Vector backend: faiss  |  NLI model: cross-encoder/nli-deberta-v3-small


In [64]:
import pprint

_summary = {
    "PDF_DIR": str(cfg.PDF_DIR),
    "PARSED_DIR": str(cfg.PARSED_DIR),
    "CHUNKS_DIR": str(cfg.CHUNKS_DIR),
    "EMBEDDING_MODEL": cfg.EMBEDDING_MODEL,
    "NLI_MODEL": cfg.NLI_MODEL,
    "CHUNK_SIZE": cfg.CHUNK_SIZE,
    "CHUNK_OVERLAP": cfg.CHUNK_OVERLAP,
    "BATCH_SIZE": cfg.BATCH_SIZE,
    "LEARNING_RATE": cfg.LEARNING_RATE,
    "EPOCHS": cfg.EPOCHS,
    "DEVICE": str(DEVICE),
}
pprint.pprint(_summary, sort_dicts=False)

{'PDF_DIR': '/home/lonewolf/quant/Projects/FINANCIAL_LLM/v3/data/raw_documents',
 'PARSED_DIR': '/home/lonewolf/quant/Projects/FINANCIAL_LLM/v3/data/parsed',
 'CHUNKS_DIR': '/home/lonewolf/quant/Projects/FINANCIAL_LLM/v3/data/chunks',
 'EMBEDDING_MODEL': 'sentence-transformers/all-mpnet-base-v2',
 'NLI_MODEL': 'cross-encoder/nli-deberta-v3-small',
 'CHUNK_SIZE': 512,
 'CHUNK_OVERLAP': 64,
 'BATCH_SIZE': 8,
 'LEARNING_RATE': 1e-05,
 'EPOCHS': 5,
 'DEVICE': 'cuda'}


In [65]:
from dataclasses import dataclass as _dc, field as _field


@_dc
class FilingRecord:
    """Lightweight record representing a single downloaded filing."""
    ticker: str
    form_type: str
    filing_date: str
    accession_number: str
    local_path: str          # absolute path to the primary document
    file_size_kb: float
    source: str              # "edgar" | "manual"
    doc_hash: str = ""       # SHA-256 of file content

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)

In [66]:
from sec_edgar_downloader import Downloader
from dataclasses import  asdict 

class SECEdgarIngester:

    def __init__(self, config: PipelineConfig)-> None:
        self.cfg = config
        self._dl = Downloader(
            company_name = "TRResearch",
            email_address = "research@thomsonresuters.com",
            download_folder = str(config.PDF_DIR),
        )
        self._records: List[FilingRecord] = []

    def download(
        self,
        tickers: List[str],
        form_types: Optional[List[str]] = None,
        after: Optional[str] = None,
        before: Optional[str] = None,
        limit: Optional[int] = None,
    ) -> List[FilingRecord]:
        form_types = form_types or list(self.cfg.SUPPORTED_FORMS)
        after = after or self.cfg.DEFAULT_AFTER_DATE
        before = before or self.cfg.DEFAULT_BEFORE_DATE
        limit = limit or self.cfg.MAX_FILINGS_PER_TICKER

        log.info(
            "Starting SEC download | tickers=%s | forms=%s | %s→%s | limit=%d",
            tickers, form_types, after, before, limit,            
        )

        for ticker in tickers:
            for form in form_types:
                self._download_one(ticker, form, after, before, limit)
        
        self._save_manifest()
        log.info("Download complete - %d filings collected", len(self._records))
        return self._records
    
    def load_manual_pdfs(
            self,
            directory: Optional[Path] = None,
            ticker_override: str = "UNKNOWN",
            form_override: str = "UNKNOWN",
    )-> List[FilingRecord]:
        directory = directory or self.cfg.PDF_DIR
        existing_paths = {r.local_path for r in self._records}
        new_records: List[FilingRecord] = []

        for pdf_path in sorted(directory.rglob("*.pdf")):
            if str(pdf_path) in existing_paths:
                continue

            parts = pdf_path.parts
            ticker = self._infer_ticker(parts) or ticker_override
            form = self._infer_form(parts) or form_override
            date = self._infer_date(parts) or datetime.now().strftime("%Y-%m-%d")

            rec = FilingRecord(
                ticker=ticker,
                form_type=form,
                filing_date = date,
                accession_number=self._hash_path(pdf_path),
                local_path=str(pdf_path),
                file_size_kb=pdf_path.stat().st_size / 1024,
                source = "manual",
                doc_hash = self._sha256(pdf_path),
            )

            new_records.append(rec)
            log.debug("Manual PDF registered: %s", pdf_path.name)
        
        self._records.extend(new_records)
        if new_records:
            self._save_manifest()
        log.info("Manual PDF scan complete - %d new files registered", len(new_records))
        return new_records
    
    @property
    def records(self)-> List[FilingRecord]:
        return list(self._records)
    
    def get_manifest(self)->pd.DataFrame:

        if not self._records:
            log.warning("No records in manifest yet.")
            return pd.DataFrame()
        return pd.DataFrame([r.to_dict() for r in self._records])
    
    def _download_one(
        self,
        ticker: str,
        form: str,
        after: str,
        before: str,
        limit: int,
    )-> None:
        log.info("  ↳ %s / %s", ticker, form)
        try:
            self._dl.get(
                form,
                ticker,
                limit=limit,
                after=after,
                before=before,
                download_details = True,
            )
        except Exception as exc:
            log.warning("Download Failed for %s/%s: %s", ticker, form, exc)
            return
        
        edgar_root = self.cfg.PDF_DIR / ticker.upper() / form.replace("-", "")
        if not edgar_root.exists():
            edgar_root = self.cfg.PDF_DIR / "sec_edgar_filings" / ticker.upper() / form

        for filing_dir in sorted(edgar_root.rglob("*")) if edgar_root.exists() else []:
            if not filing_dir.is_file():
                continue
            if filing_dir.suffix not in {".htm", ".html", ".txt", ".pdf", ".xml"}:
                continue

            accession = self._extract_accession(filing_dir)
            date = self._infer_date(filing_dir.parts) or self.cfg.DEFAULT_AFTER_DATE

            rec = FilingRecord(
                ticker = ticker.upper(),
                form_type = form,
                filing_date = date,
                accession_number=accession,
                local_path=str(filing_dir),
                file_size_kb=filing_dir.stat().st_size / 1024,
                source = "edgar",
                doc_hash=self._sha256(filing_dir),
            )

            if rec.doc_hash not in {r.doc_hash for r in self._records}:
                self._records.append(rec)

    def _save_manifest(self) -> None:
        """Persist manifest to Parquet for downstream consumption."""
        manifest_path = self.cfg.DATA_DIR / "raw_document_manifest.parquet"
        self.get_manifest().to_parquet(manifest_path, index=False)
        log.debug("Manifest saved → %s", manifest_path)

    @staticmethod
    def _sha256(path: Path) -> str:
        """Return SHA-256 hex digest of a file (chunked for large files)."""
        h = hashlib.sha256()
        with path.open("rb") as fh:
            for chunk in iter(lambda: fh.read(65_536), b""):
                h.update(chunk)
        return h.hexdigest()

    @staticmethod
    def _hash_path(path: Path) -> str:
        return hashlib.md5(str(path).encode()).hexdigest()[:16]

    @staticmethod
    def _extract_accession(path: Path) -> str:
        """Extract EDGAR accession number from a file path if present."""
        # Accession numbers follow the pattern: XXXXXXXXXX-YY-ZZZZZZ
        match = re.search(r"(\d{10}-\d{2}-\d{6})", str(path))
        return match.group(1) if match else path.stem

    @staticmethod
    def _infer_ticker(parts: Tuple[str, ...]) -> Optional[str]:
        """Heuristically extract a ticker from a directory path."""
        # Ticker is typically an all-caps short string segment in the path
        for part in parts:
            if re.fullmatch(r"[A-Z]{1,5}", part):
                return part
        return None

    @staticmethod
    def _infer_form(parts: Tuple[str, ...]) -> Optional[str]:
        """Heuristically extract a form type from a directory path."""
        form_pattern = re.compile(r"^(10-?[KkQq]|8-?[Kk])$")
        for part in parts:
            if form_pattern.match(part):
                return part.upper()
        return None

    @staticmethod
    def _infer_date(parts: Tuple[str, ...]) -> Optional[str]:
        """Heuristically extract a YYYY-MM-DD date from path segments."""
        date_pattern = re.compile(r"(\d{4}-\d{2}-\d{2})")
        for part in parts:
            m = date_pattern.search(part)
            if m:
                return m.group(1)
        return None


log.info("SECEdgarIngester class defined")   

2026-07-02 19:02:04 | INFO     | hallucination_engine | SECEdgarIngester class defined


In [67]:
DEMO_TICKERS: List[str] = ["AAPL", "MSFT"]   
DEMO_FORMS: List[str] = ["10-K"]            
SKIP_EDGAR_DOWNLOAD: bool = False           

ingester = SECEdgarIngester(cfg)

if not SKIP_EDGAR_DOWNLOAD:
    filing_records = ingester.download(
        tickers=DEMO_TICKERS,
        form_types=DEMO_FORMS,
        limit=cfg.MAX_FILINGS_PER_TICKER,
    )

# Always scan for manually placed PDFs
manual_records = ingester.load_manual_pdfs()

manifest_df = ingester.get_manifest()
log.info("Total filings in manifest: %d", len(manifest_df))

2026-07-02 19:02:04 | INFO     | hallucination_engine | Starting SEC download | tickers=['AAPL', 'MSFT'] | forms=['10-K'] | 2022-01-01→2024-12-31 | limit=3
2026-07-02 19:02:04 | INFO     | hallucination_engine |   ↳ AAPL / 10-K
2026-07-02 19:02:05 | INFO     | hallucination_engine |   ↳ MSFT / 10-K
2026-07-02 19:02:05 | WARNING  | hallucination_engine | No records in manifest yet.
2026-07-02 19:02:05 | INFO     | hallucination_engine | Download complete - 0 filings collected
2026-07-02 19:02:05 | INFO     | hallucination_engine | Manual PDF scan complete - 8 new files registered
2026-07-02 19:02:05 | INFO     | hallucination_engine | Total filings in manifest: 8


In [68]:
if not manifest_df.empty:
    display_cols = ["ticker", "form_type", "filing_date", "file_size_kb", "source"]
    print(manifest_df[display_cols].to_string(index=False))
else:
    log.warning(
        "Manifest is empty — either download failed or no PDFs exist in %s",
        cfg.PDF_DIR,
    )
    print(f"To proceed, place PDF files in: {cfg.PDF_DIR}")

 ticker form_type filing_date  file_size_kb source
UNKNOWN   UNKNOWN  2026-07-02   2561.873047 manual
UNKNOWN   UNKNOWN  2026-07-02   4055.045898 manual
UNKNOWN   UNKNOWN  2026-07-02   4115.782227 manual
UNKNOWN   UNKNOWN  2026-07-02   8461.655273 manual
UNKNOWN   UNKNOWN  2026-07-02   3128.818359 manual
UNKNOWN   UNKNOWN  2026-07-02   2944.646484 manual
UNKNOWN   UNKNOWN  2026-07-02   2838.692383 manual
UNKNOWN   UNKNOWN  2026-07-02   1902.208008 manual


In [69]:
from dataclasses import dataclass as _dc2, field as _f2


@_dc2
class ParsedParagraph:
    """Atomic unit of text within a section."""
    text:str
    paragraph_index: int
    tables: List[Dict[str, Any]] = _f2(default_factory=list)
    char_count: int = 0

    def __post_init__(self) -> None:
        self.char_count = len(self.text)

@_dc2
class ParsedSection:
    """A logical section within a page (corresponds to a heading or item)."""
    section_title : str
    section_level : str
    section_index : int
    paragraphs: List[ParsedParagraph] = _f2(default_factory=list)

    @property
    def full_text(self) -> str:
        return " ".join(p.text for p in self.paragraphs)

@_dc2
class ParsedPage:
    """All content extracted from a single PDF page."""

    page_number: int
    sections: List[ParsedSection] = _f2(default_factory=list)
    raw_text: str = ""
    has_table: bool = False 

    @property
    def full_text(self) -> str:
        return " ".join(s.full_text for s in self.sections) or self.raw_text
    
@_dc2 
class DocumentMetadata:
    """Filing-level metadata preserved on every document"""
    ticker: str
    filing_type: str
    filing_date: str
    source_path: str
    total_pages: int = 0
    doc_hash: str = ""
    parse_timestamp: str = _f2(default_factory=lambda: datetime.utcnow().isoformat())

@_dc2
class ParsedDocument: 
    """Root object returned by the parser - one per filing"""
    metadata: DocumentMetadata
    pages: List[ParsedPage] = _f2(default_factory=list)

    @property
    def all_sections(self) -> Generator[ParsedSection, None, None]:
        """Flat iterator over every section across all pages."""
        for page in self.pages:
            yield from page.sections

    @property 
    def total_char_count(self) -> int:
        return sum(p.char_count for page in self.pages
                   for s in page.sections for p in s.paragraphs)
    
    def to_dict(self) -> Dict[str, Any]:
        """JSON-serialisable representation."""
        return asdict(self)
    
log.info("ParsedDocument data models defined")

2026-07-02 19:02:05 | INFO     | hallucination_engine | ParsedDocument data models defined


In [70]:
SEC_ITEM_PATTERN = re.compile(
    r"^\s*(Item\s+\d+[A-Za-z]?\.?[\s\-:])",
    re.IGNORECASE | re.MULTILINE,
)

HEADING_PATTERNS: List[Tuple[str, str]] = [
    # (regex pattern, heading level)
    (r"^\s*(ITEM\s+\d+[A-Za-z]?\.?[\s\-:].{0,120})",  "H1"),  # SEC Items
    (r"^\s*(PART\s+[IVXivx]+\.?[\s\-:].{0,60})",        "H1"),  # SEC Parts
    (r"^[A-Z][A-Z\s,&]{4,80}$",                          "H2"),  # ALL-CAPS headings
    (r"^\d+\.\s+[A-Z][^.]{5,80}$",                       "H3"),  # Numbered sub-headings
]
_compiled_heading_patterns = [
    (re.compile(p, re.MULTILINE), lvl)
    for p, lvl in HEADING_PATTERNS
]

# Font-size thresholds (points) for PyMuPDF-based heading detection
HEADING_FONT_SIZES = {"H1": 14.0, "H2": 12.0, "H3": 11.0}
BODY_FONT_SIZE_THRESHOLD = 10.5

In [71]:
import fitz
import pdfplumber 
from tqdm.auto import tqdm

class FinancialPDFParser:
    """Parse SEC filing PDFs into structured ParsedDocument objects.

    Strategy
    --------
    1. **PyMuPDF** iterates pages and extracts text blocks with font metadata.
       Font size and boldness are used to classify headings vs body text.
    2. **pdfplumber** is used on the same pages to extract table data.
    3. Text is split into paragraphs by blank-line boundaries, then assigned
       to sections based on heading detection."""
    
    def __init__(
            self, 
            config : PipelineConfig,
            min_paragraph_chars: int = 40,
    )->None:
        self.cfg = config
        self.min_paragraph_chars = min_paragraph_chars

    def parse_file(
            self,
            path: Path,
            ticker: str = "UNKNOWNN",
            filing_type: str = "UNKNOWN",
            filing_date: str = "UNKNOWN",
    )-> ParsedDocument:
        """Parse a single PDF file into a ``ParsedDocument``.
        Parameters
        ----------
        path:         Absolute path to the PDF.
        ticker:       Issuer ticker symbol.
        filing_type:  Form type, e.g. ``"10-K"``.
        filing_date:  Filing date string ``"YYYY-MM-DD"``.

        Returns
        -------
        ParsedDocument with fully populated pages / sections / paragraphs."""

        log.info("Parsing %s [%s / %s]", path.name, ticker, filing_type)
        path = Path(path)
        if not path.exists():
            raise FileNotFoundError(f"PDF not found: {path}")
        
        doc_hash = SECEdgarIngester._sha256(path)
        metadata = DocumentMetadata(
            ticker = ticker,
            filing_type=filing_type,
            filing_date=filing_date,
            source_path=str(path),
            doc_hash = doc_hash,
        )

        pages: List[ParsedPage] = []
        try:
            pages = self._parse_pdf(path)
        except Exception as exc:
            log.error("Failed to parse %s: %s", path.name, exc, exc_info=True)
            raise

        metadata.total_pages = len(pages)
        doc = ParsedDocument(metadata=metadata, pages = pages)
        log.info(
            "  ✓ %d pages | %d chars | %d sections",
            len(pages),
            doc.total_char_count,
            sum(1 for _ in doc.all_sections),           
        )
        return doc
    
    def parse_manifest(
            self,
            manifest: pd.DataFrame,
            save: bool = True,
    )-> List[ParsedDocument]:
        """Parse every PDF-like file in *manifest*.

        Parameters
        ----------
        manifest:  DataFrame as produced by ``SECEdgarIngester.get_manifest()``.
        save:      If True, persist each ``ParsedDocument`` to JSON.

        Returns
        -------
        List of ``ParsedDocument`` objects.
        """

        if manifest.empty:
            log.warning("Empty manifest - nothing to parse.")
            return []
        
        pdf_rows = manifest[
            manifest["local_path"].str.lower().str.endswith(".pdf")
        ].copy()

        log.info("Parsing %d PDF filings from manifest", len(pdf_rows))
        parsed_docs: List[ParsedDocument] = []
        parse_summary: List[Dict[str, Any]] = []

        for _, row in tqdm(pdf_rows.iterrows(), total=len(pdf_rows), desc="Parsing PDFs"):
            try:
                doc = self.parse_file(
                    path=Path(row["local_path"]),
                    ticker=row["ticker"],
                    filing_type=row["form_type"],
                    filing_date=row["filing_date"],
                )
                parsed_docs.append(doc)

                if save:
                    self._save_document(doc)

                parse_summary.append({
                    "ticker": doc.metadata.ticker,
                    "filing_type": doc.metadata.filing_type,
                    "filing_date": doc.metadata.filing_date,
                    "total_pages": doc.metadata.total_pages,
                    "total_chars": doc.total_char_count,
                    "total_sections": sum(1 for _ in doc.all_sections),
                    "output_path": str(self._doc_output_path(doc)),
                })

            except Exception as exc:
                log.error("Skipping %s: %s", row["local_path"], exc)

        # Save parsed manifest
        if parse_summary:
            summary_df = pd.DataFrame(parse_summary)
            manifest_out = self.cfg.PARSED_DIR / "parsed_manifest.parquet"
            summary_df.to_parquet(manifest_out, index=False)
            log.info("Parsed manifest saved → %s", manifest_out)

        log.info("Parsing complete — %d / %d documents succeeded",
                 len(parsed_docs), len(pdf_rows))
        return parsed_docs
    
    def _parse_pdf(self, path: Path) -> List[ParsedPage]:
        """Core extraction using PyMuPDF + pdfplumber."""
        pages: List[ParsedPage] = []

        # Open with both libraries (pdfplumber wraps PyMuPDF internally
        # but provides a cleaner table API)
        with fitz.open(str(path)) as mupdf_doc, \
             pdfplumber.open(str(path)) as plumber_doc:

            n_pages = min(len(mupdf_doc), len(plumber_doc.pages))

            for page_idx in range(n_pages):
                mu_page = mupdf_doc[page_idx]
                pl_page = plumber_doc.pages[page_idx]

                # 1. Extract raw text blocks with font metadata
                blocks = self._extract_text_blocks(mu_page)

                # 2. Extract tables via pdfplumber
                tables = self._extract_tables(pl_page)

                # 3. Build sections from blocks
                sections = self._build_sections(blocks, tables, page_idx)

                # 4. Raw text fallback (for pages that yield no blocks)
                raw_text = mu_page.get_text("text") if not sections else ""

                parsed_page = ParsedPage(
                    page_number=page_idx + 1,
                    sections=sections,
                    raw_text=self._clean_text(raw_text),
                    has_table=len(tables) > 0,
                )
                pages.append(parsed_page)

        return pages

    def _extract_text_blocks(
        self, page: fitz.Page
    ) -> List[Dict[str, Any]]:
        """Extract text blocks from a PyMuPDF page with font metadata.

        Returns a list of dicts with keys:
        ``text``, ``font_size``, ``is_bold``, ``bbox``, ``block_type``.
        """
        blocks: List[Dict[str, Any]] = []

        # get_text("dict") returns blocks → lines → spans with font info
        page_dict = page.get_text("dict", flags=fitz.TEXT_PRESERVE_WHITESPACE)

        for block in page_dict.get("blocks", []):
            if block.get("type") != 0:   # 0 = text, 1 = image
                continue

            block_text_parts: List[str] = []
            font_sizes: List[float] = []
            is_bold = False

            for line in block.get("lines", []):
                for span in line.get("spans", []):
                    span_text = span.get("text", "").strip()
                    if span_text:
                        block_text_parts.append(span_text)
                        font_sizes.append(span.get("size", 10.0))
                        flags = span.get("flags", 0)
                        # Bit 4 (value 16) = bold in PDF font flags
                        if flags & 16:
                            is_bold = True

            combined_text = " ".join(block_text_parts)
            if not combined_text.strip():
                continue

            avg_font_size = float(np.mean(font_sizes)) if font_sizes else 10.0

            blocks.append({
                "text": self._clean_text(combined_text),
                "font_size": avg_font_size,
                "is_bold": is_bold,
                "bbox": block.get("bbox", []),
                "block_type": self._classify_block(
                    combined_text, avg_font_size, is_bold
                ),
            })

        return blocks

    def _extract_tables(
        self, page: pdfplumber.page.Page
    ) -> List[Dict[str, Any]]:
        """Extract tables from a pdfplumber page.

        Returns a list of dicts with keys ``rows`` (list of row lists)
        and ``bbox``.
        """
        tables: List[Dict[str, Any]] = []
        try:
            for table in page.extract_tables():
                if not table:
                    continue
                # Normalise cells: strip whitespace, replace None
                rows = [
                    [str(cell).strip() if cell is not None else "" for cell in row]
                    for row in table
                ]
                tables.append({"rows": rows})
        except Exception as exc:
            log.debug("Table extraction warning: %s", exc)
        return tables

    def _build_sections(
        self,
        blocks: List[Dict[str, Any]],
        tables: List[Dict[str, Any]],
        page_idx: int,
    ) -> List[ParsedSection]:
        """Group text blocks into sections using heading detection.

        A new section is started whenever a heading block is encountered.
        All subsequent body blocks belong to the current section until
        the next heading.
        """
        if not blocks:
            return []

        sections: List[ParsedSection] = []
        current_section = ParsedSection(
            section_title="[Page Start]",
            section_level="body",
            section_index=0,
        )
        para_idx = 0
        section_idx = 0

        for block in blocks:
            block_type = block["block_type"]
            text = block["text"]

            if block_type in {"H1", "H2", "H3"}:
                # Save current section if it has content
                if current_section.paragraphs:
                    sections.append(current_section)
                section_idx += 1
                current_section = ParsedSection(
                    section_title=text[:200],   # cap title length
                    section_level=block_type,
                    section_index=section_idx,
                )
                para_idx = 0
            else:
                # Body text — split into paragraphs on double newline
                raw_paras = re.split(r"\n{2,}", text)
                for raw_para in raw_paras:
                    clean = self._clean_text(raw_para)
                    if len(clean) < self.min_paragraph_chars:
                        continue
                    para = ParsedParagraph(
                        text=clean,
                        paragraph_index=para_idx,
                    )
                    current_section.paragraphs.append(para)
                    para_idx += 1

        # Don't forget the last section
        if current_section.paragraphs:
            sections.append(current_section)

        # Attach tables to the last section on the page (heuristic)
        if tables and sections:
            last_section = sections[-1]
            if last_section.paragraphs:
                last_section.paragraphs[-1].tables = tables

        return sections

    def _classify_block(
        self,
        text: str,
        font_size: float,
        is_bold: bool,
    ) -> str:
        """Classify a text block as H1/H2/H3/body using font + regex heuristics.

        Priority order:
        1. SEC Item/Part pattern (most reliable for 10-K/10-Q)
        2. Font size thresholds
        3. ALL-CAPS regex patterns
        4. Default to body
        """
        # 1. SEC-specific patterns
        if SEC_ITEM_PATTERN.match(text):
            return "H1"

        # 2. Font-size heuristic
        if font_size >= HEADING_FONT_SIZES["H1"] and is_bold:
            return "H1"
        if font_size >= HEADING_FONT_SIZES["H2"] and is_bold:
            return "H2"
        if font_size >= HEADING_FONT_SIZES["H3"] and is_bold:
            return "H3"

        # 3. Text pattern heuristics (for non-bold PDFs)
        for pattern, level in _compiled_heading_patterns:
            if pattern.match(text):
                return level

        return "body"
    
    # text cleaning
    @staticmethod
    def _clean_text(text: str) -> str:
        """Normalise extracted PDF text.

        - Collapse multiple spaces / tabs to a single space.
        - Normalise various dash / hyphen Unicode variants.
        - Remove PDF artefacts (form-feed, null bytes).
        - Strip leading/trailing whitespace.
        """
        if not text:
            return ""
        # Remove null bytes and form feeds
        text = text.replace("\x00", "").replace("\x0c", " ")
        # Normalise dash variants
        text = re.sub(r"[\u2013\u2014\u2015]", "-", text)
        # Normalise whitespace (keep single newlines, collapse multiples)
        text = re.sub(r"[ \t]+", " ", text)
        text = re.sub(r"\n{3,}", "\n\n", text)
        # Remove soft-hyphen continuation artefacts
        text = re.sub(r"\u00ad", "", text)
        return text.strip()

    def _doc_output_path(self, doc: ParsedDocument) -> Path:
        safe_ticker = re.sub(r"[^\w]", "_", doc.metadata.ticker)
        safe_form = re.sub(r"[^\w]", "_", doc.metadata.filing_type)
        safe_date = re.sub(r"[^\w]", "_", doc.metadata.filing_date)
        filename = f"{safe_ticker}_{safe_form}_{safe_date}.json"
        return self.cfg.PARSED_DIR / filename

    def _save_document(self, doc: ParsedDocument) -> Path:
        """Serialise a ParsedDocument to JSON."""
        out_path = self._doc_output_path(doc)
        with out_path.open("w", encoding="utf-8") as fh:
            json.dump(doc.to_dict(), fh, ensure_ascii=False, indent=2)
        log.debug("Saved parsed doc → %s", out_path.name)
        return out_path


log.info("FinancialPDFParser class defined")

2026-07-02 19:02:05 | INFO     | hallucination_engine | FinancialPDFParser class defined


In [72]:
# Utility function to load a ParsedDocument

def load_parsed_document(json_path: Path) -> ParsedDocument:
    """Deserialise a ParsedDocument from a JSON file saved by the parser.

    This function is used by Section 5 (chunking) to consume parser outputs
    without re-running the parser.
    """
    raw = json.loads(json_path.read_text(encoding="utf-8"))

    pages = []
    for p in raw.get("pages", []):
        sections = []
        for s in p.get("sections", []):
            paragraphs = [
                ParsedParagraph(
                    text=para["text"],
                    paragraph_index=para["paragraph_index"],
                    tables=para.get("tables", []),
                    char_count=para.get("char_count", len(para["text"])),
                )
                for para in s.get("paragraphs", [])
            ]
            sections.append(ParsedSection(
                section_title=s["section_title"],
                section_level=s["section_level"],
                section_index=s["section_index"],
                paragraphs=paragraphs,
            ))
        pages.append(ParsedPage(
            page_number=p["page_number"],
            sections=sections,
            raw_text=p.get("raw_text", ""),
            has_table=p.get("has_table", False),
        ))

    meta = raw["metadata"]
    metadata = DocumentMetadata(
        ticker=meta["ticker"],
        filing_type=meta["filing_type"],
        filing_date=meta["filing_date"],
        source_path=meta["source_path"],
        total_pages=meta.get("total_pages", len(pages)),
        doc_hash=meta.get("doc_hash", ""),
        parse_timestamp=meta.get("parse_timestamp", ""),
    )
    return ParsedDocument(metadata=metadata, pages=pages)


log.info("load_parsed_document helper defined")


2026-07-02 19:02:05 | INFO     | hallucination_engine | load_parsed_document helper defined


In [73]:
# Running Parseing Pipeline 

SKIP_PARSING: bool = False
parser = FinancialPDFParser(cfg)

if SKIP_PARSING:
    parsed_docs = [
        load_parsed_document(p)
        for p in sorted(cfg.PARSED_DIR.glob("*.json"))
    ]

    log.info("Loaded %d pre-parsed documents from %s", len(parsed_docs), cfg.PARSED_DIR)
else:
    parsed_docs = parser.parse_manifest(manifest_df, save=True)

log.info("ParsedDocument objects ready.")

2026-07-02 19:02:05 | INFO     | hallucination_engine | Parsing 8 PDF filings from manifest


Parsing PDFs:   0%|          | 0/8 [00:00<?, ?it/s]

2026-07-02 19:02:05 | INFO     | hallucination_engine | Parsing AAPL_10K.pdf [UNKNOWN / UNKNOWN]
2026-07-02 19:02:12 | INFO     | hallucination_engine |   ✓ 71 pages | 181943 chars | 75 sections
2026-07-02 19:02:12 | INFO     | hallucination_engine | Parsing GOOG_10K_2025.pdf [UNKNOWN / UNKNOWN]
2026-07-02 19:02:31 | INFO     | hallucination_engine |   ✓ 110 pages | 323737 chars | 119 sections
2026-07-02 19:02:31 | INFO     | hallucination_engine | Parsing GOOG_10K_2026.pdf [UNKNOWN / UNKNOWN]
2026-07-02 19:02:46 | INFO     | hallucination_engine |   ✓ 107 pages | 316940 chars | 116 sections
2026-07-02 19:02:46 | INFO     | hallucination_engine | Parsing MSFT_10K.pdf [UNKNOWN / UNKNOWN]
2026-07-02 19:03:05 | INFO     | hallucination_engine |   ✓ 102 pages | 286830 chars | 134 sections
2026-07-02 19:03:05 | INFO     | hallucination_engine | Parsing NVDA_10K_2025.pdf [UNKNOWN / UNKNOWN]
2026-07-02 19:03:13 | INFO     | hallucination_engine |   ✓ 87 pages | 326580 chars | 105 sections
202

In [74]:
# Inspection

def summarise_parsed_document(doc: ParsedDocument) -> pd.DataFrame:
    """Return a section-level summary DataFrame for a ParsedDocument."""

    rows = []
    for page in doc.pages:
        for section in page.sections:
            rows.append({
                "page" : page.page_number,
                "level" : section.section_level,
                "title" : section.section_title[:80],
                "paragraphs" : len(section.paragraphs),
                "chars" : sum(p.char_count for p in section.paragraphs),
                "tables" : sum(len(p.tables) for p in section.paragraphs),
            })
    return pd.DataFrame(rows)

if parsed_docs:
    demo_doc = parsed_docs[0]
    print(f"\n{'='*70}")
    print(f" Document: {demo_doc.metadata.ticker} / "
          f"{demo_doc.metadata.filing_type} / "
          f"{demo_doc.metadata.filing_date}")
    print(f" Pages: {demo_doc.metadata.total_pages} | "
          f"Total chars: {demo_doc.total_char_count:,}")
    print(f"{'='*70}")

    summary_df = summarise_parsed_document(demo_doc)
    if not summary_df.empty:
        print(summary_df.to_string(index=False))
    else:
        print("No sections detected — check PDF structure or heading patterns.")
else:
    print("No documents parsed yet. Check ingestion step.")


 Document: UNKNOWN / UNKNOWN / 2026-07-02
 Pages: 71 | Total chars: 181,943
 page level                                                                            title  paragraphs  chars  tables
    1    H1                                                                        FORM 10-K           4    275       0
    1    H1                                                                       Apple Inc.          19   1369       0
    2  body                                                                     [Page Start]          14   2722       0
    2    H2                                              DOCUMENTS INCORPORATED BY REFERENCE           1    389       0
    3  body                                                                     [Page Start]           1     44       0
    3    H1                                                                        Item 12 .           1     86       6
    4  body                                                                     [Pa

In [75]:
from transformers import AutoTokenizer
from tqdm.auto import tqdm 

@dataclass
class TextChunk:
    """A single retrieval unit produced by the chunker."""
    chunk_id: str; text: str
    ticker: str; filing_type: str; filing_date: str
    page: int; section_title: str; section_level: str; section_index: int
    token_count: int; char_count: int; source_path: str; doc_hash: str
    chunk_index: int
    def to_dict(self): return asdict(self)

class SectionAwareChunker:
    """Token-bounded chunks that never cross section boundaries.

    Uses the embedding model's tokenizer so CHUNK_SIZE is exact in token space.
    Employs a sliding window with configurable overlap.
    """

    def __init__(self, config: PipelineConfig, tokenizer_name: Optional[str] = None):
        self.cfg = config
        mn = tokenizer_name or config.EMBEDDING_MODEL
        log.info('Loading chunker tokenizer: %s', mn)
        self.tokenizer = AutoTokenizer.from_pretrained(mn)
        self.chunk_size = config.CHUNK_SIZE
        self.chunk_overlap = config.CHUNK_OVERLAP

    def chunk_documents(self, documents: List[ParsedDocument], save: bool = True)->pd.DataFrame:
        """Chunk all documents and return a DataFrame of TextChunk rows."""
        all_chunks: List[TextChunk] = []
        for doc in tqdm(documents, desc='Chunking documents'):
            doc_chunks = self._chunk_document(doc)
            all_chunks.extend(doc_chunks)
            log.debug('%s/%s -> %d chunks', doc.metadata.ticker, doc.metadata.filing_type, len(doc_chunks))

        if not all_chunks:
            log.warning("No chunks produced - verify parsed_doc is non-empty.")
            return pd.DataFrame()

        df = pd.DataFrame([c.to_dict() for c in all_chunks])
        log.info("Total chunks: %d | Mean tokens: %.1f | Tickers: %s",
                 len(df), df['token_count'].mean(), df['ticker'].nunique())
        
        if save:
            out = self.cfg.CHUNKS_DIR / 'chunks.parquet'
            df.to_parquet(out, index= False)
            log.info('Chunks saved -> %s (%d rows)', out, len(df))

        return df
    
    def _chunk_document(self, doc: ParsedDocument) -> List[TextChunk]:
        chunks: List[TextChunk] = []
        meta = doc.metadata
        for page in doc.pages:
            for section in page.sections:
                text = section.full_text
                if not text.strip(): continue
                chunks.extend(self._chunk_text(
                    text=text, ticker=meta.ticker, filing_type=meta.filing_type,
                    filing_date=meta.filing_date, page=page.page_number,
                    section_title=section.section_title, section_level=section.section_level,
                    section_index=section.section_index, source_path=meta.source_path,
                    doc_hash=meta.doc_hash))
        return chunks

    def _chunk_text(self, text: str, **meta) -> List[TextChunk]:
        """Sliding window over token IDs, decoding back to strings."""
        ids = self.tokenizer.encode(text, add_special_tokens=False, truncation=False)
        if not ids: return []
        step = max(1, self.chunk_size - self.chunk_overlap)
        chunks, chunk_idx, start = [], 0, 0
        while start < len(ids):
            end = min(start + self.chunk_size, len(ids))
            window = ids[start:end]
            chunk_text = self.tokenizer.decode(
                window, skip_special_tokens=True, clean_up_tokenization_spaces=True).strip()
            if chunk_text:
                cid = hashlib.sha1(
                    f"{meta.get('ticker')}|{meta.get('filing_date')}|"
                    f"{meta.get('section_index')}|{chunk_idx}".encode()).hexdigest()[:16]
                chunks.append(TextChunk(
                    chunk_id=cid, text=chunk_text, token_count=len(window),
                    char_count=len(chunk_text), chunk_index=chunk_idx, **meta))
                chunk_idx += 1
            if end == len(ids): break
            start += step
        return chunks


log.info('SectionAwareChunker defined')

2026-07-02 19:03:38 | INFO     | hallucination_engine | SectionAwareChunker defined


In [76]:
SKIP_CHUNKING: bool = False
CHUNKS_PATH: Path = cfg.CHUNKS_DIR / 'chunks.parquet'

if SKIP_CHUNKING and CHUNKS_PATH.exists():
    chunks_df = pd.read_parquet(CHUNKS_PATH)
    log.info('Loaded %d chunks from %s', len(chunks_df), CHUNKS_PATH)
else:
    # Synthetic fallback when no real parsed docs exist
    if not parsed_docs:
        log.warning('No parsed documents — building synthetic corpus for demo.')
        SYNTHETIC = [
            ('AAPL','10-K','2023-11-03','Item 1. Business','H1',
             'Apple Inc. designs, manufactures, and markets smartphones, personal computers, '
             'tablets, wearables, and accessories. Net sales for fiscal 2023 were $383.3 billion, '
             'a decrease of 2.8 percent compared to fiscal 2022.'),
            ('AAPL','10-K','2023-11-03','Item 1A. Risk Factors','H1',
             'The markets for the Company\'s products and services are highly competitive. '
             'The Company has substantial exposure to risks associated with manufacturing in China.'),
            ('AAPL','10-K','2023-11-03','Item 7. MD&A','H1',
             'Revenue from iPhone was $200.6 billion, representing 52 percent of total net sales. '
             'Services revenue reached a record $85.2 billion, growing 9 percent year-over-year.'),
            ('MSFT','10-K','2023-07-27','Item 1. Business','H1',
             'Microsoft Corporation enables digital transformation. '
             'Revenue for fiscal 2023 was $211.9 billion, representing 7 percent growth. '
             'Cloud revenue including Azure was $111.6 billion.'),
            ('MSFT','10-K','2023-07-27','Item 7. MD&A','H1',
             'Azure and other cloud services revenue grew 29 percent. '
             'Operating income was $88.5 billion. Net income was $72.4 billion.'),
        ]
        for ticker,form,date,title,level,text in SYNTHETIC:
            sec = ParsedSection(section_title=title, section_level=level, section_index=0,
                                paragraphs=[ParsedParagraph(text=text, paragraph_index=0)])
            page = ParsedPage(page_number=1, sections=[sec])
            parsed_docs.append(ParsedDocument(
                metadata=DocumentMetadata(ticker=ticker, filing_type=form, filing_date=date,
                    source_path='synthetic', doc_hash=hashlib.md5(ticker.encode()).hexdigest()),
                pages=[page]))
        log.info('Synthetic corpus: %d documents', len(parsed_docs))

    chunker = SectionAwareChunker(cfg)
    chunks_df = chunker.chunk_documents(parsed_docs, save=True)

print(f'chunks_df shape: {chunks_df.shape}')
print(chunks_df[['ticker','section_title','token_count','chunk_id']].head(6).to_string(index=False))


2026-07-02 19:03:38 | INFO     | hallucination_engine | Loading chunker tokenizer: sentence-transformers/all-mpnet-base-v2


Chunking documents:   0%|          | 0/8 [00:00<?, ?it/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (583 > 512). Running this sequence through the model will result in indexing errors


2026-07-02 19:03:40 | INFO     | hallucination_engine | Total chunks: 1242 | Mean tokens: 351.0 | Tickers: 1
2026-07-02 19:03:40 | INFO     | hallucination_engine | Chunks saved -> /home/lonewolf/quant/Projects/FINANCIAL_LLM/v3/data/chunks/chunks.parquet (1242 rows)
chunks_df shape: (1242, 14)
 ticker                       section_title  token_count         chunk_id
UNKNOWN                           FORM 10-K           54 ee15d7d741df5b42
UNKNOWN                          Apple Inc.          354 fb05350d36ef47b2
UNKNOWN                        [Page Start]          512 bdbabfeb71c4a8e4
UNKNOWN                        [Page Start]          135 c7a8fb0e74507ee2
UNKNOWN DOCUMENTS INCORPORATED BY REFERENCE           77 c50ab8e798fc3f80
UNKNOWN                        [Page Start]           10 bdbabfeb71c4a8e4


In [77]:
if not chunks_df.empty:
    import matplotlib; matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    axes[0].hist(chunks_df['token_count'], bins=30, color='steelblue', edgecolor='white')
    axes[0].axvline(cfg.CHUNK_SIZE, color='red', linestyle='--', label=f'limit={cfg.CHUNK_SIZE}')
    axes[0].set_title('Token count distribution'); axes[0].legend()
    tc = chunks_df['ticker'].value_counts()
    axes[1].bar(tc.index, tc.values, color='teal')
    axes[1].set_title('Chunks per ticker')
    lc = chunks_df['section_level'].value_counts()
    axes[2].pie(lc.values, labels=lc.index, autopct='%1.0f%%')
    axes[2].set_title('Section level mix')
    plt.tight_layout()
    out = cfg.OUTPUTS_DIR / 'chunk_diagnostics.png'
    plt.savefig(out, dpi=120, bbox_inches='tight'); plt.show()
    log.info('Diagnostics saved → %s', out)

2026-07-02 19:03:40 | INFO     | hallucination_engine | Diagnostics saved → /home/lonewolf/quant/Projects/FINANCIAL_LLM/v3/outputs/chunk_diagnostics.png


In [78]:
from sentence_transformers import SentenceTransformer

class EmbeddingGenerator:
    """Batch-embed chunks with sentence-transformers; persist to disk."""

    _EMBED_COL = 'text'
    _META_COLS = ['chunk_id','ticker','filing_type','filing_date','page',
                  'section_title','section_level','token_count',
                  'chunk_index','source_path','doc_hash']
    
    def __init__(self, config: PipelineConfig, model_name: Optional[str]=None):
        self.cfg = config
        mn = model_name or config.EMBEDDING_MODEL
        log.info('Loading embedding model: %s', mn)
        self.model = SentenceTransformer(mn, device=str(DEVICE))
        self.embedding_dim = self.model.get_sentence_embedding_dimension()
        log.info('Embedding dim: %d', self.embedding_dim)

    def generate(self, chunks: pd.DataFrame, batch_size: Optional[int] = None, 
                 save: bool = True) -> Tuple[np.ndarray, pd.DataFrame]:
        if chunks.empty:
            raise ValueError('chunks DataFrame is empty')
        batch_size = batch_size or (self.cfg.BATCH_SIZE * 4)
        texts = chunks[self._EMBED_COL].tolist()
        n = len(texts)
        log.info('Embedding %d chunks | batch_size=%d', n, batch_size)
        t0 = time.time()

        embeddings = self.model.encode(
            texts, batch_size=batch_size, show_progress_bar=True,
            normalize_embeddings=True, convert_to_numpy=True
        )

        elapsed = time.time() - t0
        log.info('Embedded in %.1fs(%.0f chunks/s)', elapsed, n/elapsed)

        present = [c for c in self._META_COLS if c in chunks.columns]
        meta_df = chunks[present].reset_index(drop=True).copy()
        meta_df['embedding_index'] = np.arange(n, dtype=np.int64)

        if save:
            ep = self.cfg.EMBEDDINGS_DIR / 'embeddings.npy'
            mp = self.cfg.EMBEDDINGS_DIR / 'metadata.paarquet'
            np.save(str(ep), embeddings.astype(np.float32))
            meta_df.to_parquet(mp, index=False)
            log.info('Saved embeddings → %s  shape=%s', ep, embeddings.shape)

        return embeddings.astype(np.float32), meta_df

    def embed_query(self, text: str) -> np.ndarray:
        return self.model.encode(
            text, normalize_embeddings=True, convert_to_numpy=True)

    @classmethod
    def load(cls, config: PipelineConfig) -> Tuple[np.ndarray, pd.DataFrame]:
        ep = config.EMBEDDINGS_DIR / 'embeddings.npy'
        mp = config.EMBEDDINGS_DIR / 'metadata.parquet'
        if not ep.exists():
            raise FileNotFoundError(f'Embeddings not found: {ep}')
        emb = np.load(str(ep)).astype(np.float32)
        meta = pd.read_parquet(mp)
        log.info('Loaded embeddings %s', emb.shape)
        return emb, meta


log.info('EmbeddingGenerator defined')

2026-07-02 19:03:40 | INFO     | hallucination_engine | EmbeddingGenerator defined


In [79]:
SKIP_EMBEDDING: bool = False

embedder = EmbeddingGenerator(cfg)

if SKIP_EMBEDDING and (cfg.EMBEDDINGS_DIR / 'embeddings.npy').exists():
    embeddings, embed_meta_df = EmbeddingGenerator.load(cfg)
else:
    embeddings, embed_meta_df = embedder.generate(chunks_df, save=True)

print(f'Embeddings shape  : {embeddings.shape}')
print(f'L2 norms (first 5): {np.linalg.norm(embeddings[:5], axis=1).round(4)}')
print(f'Metadata columns  : {list(embed_meta_df.columns)}')

2026-07-02 19:03:40 | INFO     | hallucination_engine | Loading embedding model: sentence-transformers/all-mpnet-base-v2
2026-07-02 19:03:43 | INFO     | hallucination_engine | Embedding dim: 768
2026-07-02 19:03:43 | INFO     | hallucination_engine | Embedding 1242 chunks | batch_size=32


Batches:   0%|          | 0/39 [00:00<?, ?it/s]

2026-07-02 19:03:54 | INFO     | hallucination_engine | Embedded in 10.8s(116 chunks/s)
2026-07-02 19:03:54 | INFO     | hallucination_engine | Saved embeddings → /home/lonewolf/quant/Projects/FINANCIAL_LLM/v3/data/embeddings/embeddings.npy  shape=(1242, 768)
Embeddings shape  : (1242, 768)
L2 norms (first 5): [1. 1. 1. 1. 1.]
Metadata columns  : ['chunk_id', 'ticker', 'filing_type', 'filing_date', 'page', 'section_title', 'section_level', 'token_count', 'chunk_index', 'source_path', 'doc_hash', 'embedding_index']


In [80]:
import faiss
import chromadb
from chromadb.config import Settings as ChromaSettings

@dataclass
class RetrievalResult:
    chunk_id: str; text: str; score: float
    ticker: str; filing_type: str; filing_date: str
    page: int; section_title: str; section_level: str; rank: int
    def to_dict(self): return asdict(self)


class VectorStore:
    """Unified FAISS / ChromaDB interface with metadata filtering."""

    _CHROMA_COLLECTION = 'sec_filings'

    def __init__(self, config: PipelineConfig, backend: Optional[str] = None):
        self.cfg = config
        self.backend = (backend or config.VECTOR_BACKEND).lower()
        assert self.backend in {'faiss','chroma'}
        self._faiss_index = None; self._faiss_meta = None; self._chroma_col = None

    def build(self, embeddings: np.ndarray, metadata: pd.DataFrame,
              chunks: pd.DataFrame) -> 'VectorStore':
        """Index all embeddings and persist to disk."""
        if self.backend == 'faiss':
            self._build_faiss(embeddings, metadata, chunks)
        else:
            self._build_chroma(embeddings, metadata, chunks)
        return self

    @classmethod
    def load(cls, config: PipelineConfig) -> 'VectorStore':
        store = cls(config)
        if store.backend == 'faiss':
            ip = config.VECTOR_DB_PATH / 'faiss.index'
            mp = config.VECTOR_DB_PATH / 'faiss_meta.parquet'
            if ip.exists():
                store._faiss_index = faiss.read_index(str(ip))
                store._faiss_meta  = pd.read_parquet(mp)
                log.info('FAISS loaded — %d vectors', store._faiss_index.ntotal)
        else:
            cp = str(config.VECTOR_DB_PATH / 'chroma')
            client = chromadb.PersistentClient(path=cp,
                settings=ChromaSettings(anonymized_telemetry=False))
            store._chroma_col = client.get_collection(cls._CHROMA_COLLECTION)
        return store

    def search(self, query_vector: np.ndarray, top_k: int = 5,
               ticker_filter: Optional[str] = None,
               date_after: Optional[str] = None,
               date_before: Optional[str] = None) -> List[RetrievalResult]:
        if self.backend == 'faiss':
            return self._search_faiss(query_vector, top_k, ticker_filter, date_after, date_before)
        return self._search_chroma(query_vector, top_k, ticker_filter, date_after, date_before)

    # FAISS 
    def _build_faiss(self, emb, meta, chunks):
        dim = emb.shape[1]
        log.info('Building FAISS IndexFlatIP dim=%d n=%d', dim, len(emb))
        # assign indices to the flat indices
        index = faiss.IndexIDMap2(faiss.IndexFlatIP(dim))
        index.add_with_ids(emb.astype(np.float32), np.arange(len(emb), dtype=np.int64))
        self._faiss_index = index
        self._faiss_meta  = meta.copy()
        if 'text' not in self._faiss_meta.columns and 'text' in chunks.columns:
            self._faiss_meta = self._faiss_meta.merge(chunks[['chunk_id','text']], on='chunk_id', how='left')
        faiss.write_index(index, str(self.cfg.VECTOR_DB_PATH / 'faiss.index'))
        self._faiss_meta.to_parquet(self.cfg.VECTOR_DB_PATH / 'faiss_meta.parquet', index=False)
        log.info('FAISS saved — %d vectors', index.ntotal)

    def _search_faiss(self, qv, top_k, ticker, after, before):
        if self._faiss_index is None or self._faiss_index.ntotal == 0: return []
        # retrieve topk scores
        fetch_k = min(top_k * 10, self._faiss_index.ntotal)
        scores, indices = self._faiss_index.search(qv.reshape(1,-1).astype(np.float32), fetch_k)
        results = []
        for s, idx in zip(scores[0], indices[0]):
            if idx < 0 or idx >= len(self._faiss_meta): continue
            row = self._faiss_meta.iloc[idx]
            if ticker and str(row.get('ticker','')) != ticker: continue
            d = str(row.get('filing_date',''))
            if after and d < after: continue
            if before and d > before: continue
            results.append(RetrievalResult(
                chunk_id=str(row.get('chunk_id','')), text=str(row.get('text','')),
                score=float(s), ticker=str(row.get('ticker','')),
                filing_type=str(row.get('filing_type','')), filing_date=str(row.get('filing_date','')),
                page=int(row.get('page',0)), section_title=str(row.get('section_title','')),
                section_level=str(row.get('section_level','')), rank=len(results)+1))
            if len(results) == top_k: break
        return results

    # ChromaDB
    def _build_chroma(self, emb, meta, chunks):
        cp = str(self.cfg.VECTOR_DB_PATH / 'chroma')
        log.info('Building ChromaDB at %s', cp)
        client = chromadb.PersistentClient(path=cp,
            settings=ChromaSettings(anonymized_telemetry=False))
        try: client.delete_collection(self._CHROMA_COLLECTION)
        except Exception: pass
        col = client.create_collection(self._CHROMA_COLLECTION,
            metadata={'hnsw:space': 'cosine'})
        self._chroma_col = col
        full = meta.copy()
        if 'text' not in full.columns and 'text' in chunks.columns:
            full = full.merge(chunks[['chunk_id','text']], on='chunk_id', how='left')
        BATCH = 500
        for start in tqdm(range(0, len(full), BATCH), desc='Indexing Chroma'):
            b = full.iloc[start:start+BATCH]
            col.upsert(
                ids=[str(r) for r in b['chunk_id']],
                embeddings=emb[start:start+BATCH].tolist(),
                documents=b['text'].fillna('').tolist(),
                metadatas=b.drop(columns=['text'],errors='ignore')
                           .fillna('').astype(str).to_dict(orient='records'))
        log.info('ChromaDB built — %d docs', col.count())

    def _search_chroma(self, qv, top_k, ticker, after, before):
        where: Dict = {}
        if ticker: where['ticker'] = ticker
        kw: Dict = dict(query_embeddings=[qv.tolist()], n_results=top_k,
                        include=['documents','metadatas','distances'])
        if where: kw['where'] = where
        res = self._chroma_col.query(**kw)
        return [
            RetrievalResult(
                chunk_id=m.get('chunk_id',''), text=doc, score=1.0-float(d),
                ticker=m.get('ticker',''), filing_type=m.get('filing_type',''),
                filing_date=m.get('filing_date',''), page=int(m.get('page',0)),
                section_title=m.get('section_title',''),
                section_level=m.get('section_level',''), rank=rank)
            for rank,(doc,m,d) in enumerate(
                zip(res['documents'][0],res['metadatas'][0],res['distances'][0]),1)
        ]


log.info('VectorStore defined (faiss + chroma)')


2026-07-02 19:03:54 | INFO     | hallucination_engine | VectorStore defined (faiss + chroma)


In [81]:
SKIP_INDEXING: bool = False

_idx_file = (cfg.VECTOR_DB_PATH / 'faiss.index') if cfg.VECTOR_BACKEND == 'faiss' \
             else (cfg.VECTOR_DB_PATH / 'chroma')

if SKIP_INDEXING and _idx_file.exists():
    vector_store = VectorStore.load(cfg)
else:
    vector_store = VectorStore(cfg).build(
        embeddings=embeddings, metadata=embed_meta_df, chunks=chunks_df)

log.info('Vector store ready — backend=%s', cfg.VECTOR_BACKEND)

2026-07-02 19:03:54 | INFO     | hallucination_engine | Building FAISS IndexFlatIP dim=768 n=1242
2026-07-02 19:03:58 | INFO     | hallucination_engine | FAISS saved — 1242 vectors
2026-07-02 19:03:58 | INFO     | hallucination_engine | Vector store ready — backend=faiss


In [82]:
class RetrievalPipeline:
    """Claim → evidence retrieval with optional cross-encoder reranking."""

    def __init__(self, embedder: EmbeddingGenerator, store: VectorStore,
                 config: PipelineConfig, use_reranker: bool = False):
        self.embedder  = embedder
        self.store     = store
        self.cfg       = config
        self._reranker = None
        if use_reranker:
            try:
                from sentence_transformers import CrossEncoder
                self._reranker = CrossEncoder(
                    config.RERANKER_MODEL, device=str(DEVICE), max_length=512)
                log.info('Cross-encoder reranker loaded')
            except Exception as exc:
                log.warning('Reranker unavailable: %s', exc)

    def retrieve(self, claim: str, top_k: Optional[int] = None,
                 ticker_filter: Optional[str] = None,
                 date_after: Optional[str] = None,
                 date_before: Optional[str] = None) -> List[RetrievalResult]:
        """Retrieve top-k evidence chunks for a claim."""
        top_k = top_k or self.cfg.TOP_K
        if not claim.strip():
            raise ValueError('claim must be non-empty')
        qv = self.embedder.embed_query(claim)
        fetch_k = top_k * 3 if self._reranker else top_k
        results = self.store.search(qv, top_k=fetch_k,
            ticker_filter=ticker_filter, date_after=date_after, date_before=date_before)
        if self._reranker and results:
            pairs = [[claim, r.text] for r in results]
            scores = self._reranker.predict(pairs, show_progress_bar=False)
            for r, s in zip(results, scores): r.score = float(s)
            results = sorted(results, key=lambda r: r.score, reverse=True)[:top_k]
        for i, r in enumerate(results, 1): r.rank = i
        return results

    def retrieve_batch(self, claims: List[str], top_k: Optional[int] = None,
                       **kwargs) -> List[List[RetrievalResult]]:
        return [self.retrieve(c, top_k=top_k, **kwargs)
                for c in tqdm(claims, desc='Retrieving')]

    def format_evidence(self, results: List[RetrievalResult], max_length: int = 400) -> str:
        lines = []
        for r in results:
            snip = r.text[:max_length] + ('...' if len(r.text) > max_length else '')
            lines.append(
                f'[{r.rank}] ({r.ticker}·{r.filing_type}·{r.filing_date}·'
                f'p{r.page}·§{r.section_title[:50]}) score={r.score:.3f}\n{snip}')
        return '\n\n'.join(lines) or '(no evidence found)'


retrieval_pipeline = RetrievalPipeline(
    embedder=embedder, store=vector_store, config=cfg, use_reranker=False)
log.info('RetrievalPipeline ready')

# Smoke test 
DEMO_CLAIM = 'Apple iPhone revenue was $200.6 billion in fiscal year 2023.'
evidence_results = retrieval_pipeline.retrieve(DEMO_CLAIM, top_k=cfg.TOP_K)
print(f'Query: "{DEMO_CLAIM}"\n')
print(retrieval_pipeline.format_evidence(evidence_results))


2026-07-02 19:03:58 | INFO     | hallucination_engine | RetrievalPipeline ready
Query: "Apple iPhone revenue was $200.6 billion in fiscal year 2023."

[1] (UNKNOWN·UNKNOWN·2026-07-02·p2·§[Page Start]) score=0.644
the company ’ s ability to compete successfully depends heavily on ensuring the continuing and timely introduction of innovative new products, services and technologies to the marketplace. the company designs and develops nearly the entire solution for its products, including the hardware, operating system, numerous software applications and related services. principal competitive factors importan...

[2] (UNKNOWN·UNKNOWN·2026-07-02·p2·§[Page Start]) score=0.610
the following table shows net sales by category for 2023, 2022 and 2021 ( dollars in millions ) : iphone $ 200, 583 ( 2 ) % $ 205, 489 7 % $ 191, 973 wearables, home and accessories 39, 845 ( 3 ) % 41, 241 7 % 38, 367 t otal net sales $ 383, 285 ( 3 ) % $ 394, 328 8 % $ 365, 817 ( 1 ) products net sales include amortiz

The hallucination detection model requires **human-labeled (claim, evidence)**
pairs. The Gradio UI lets annotators see retrieved evidence in real time and
assign three-way NLI labels. All annotations are stored in a SQLAlchemy-managed
database (SQLite by default; PostgreSQL-compatible schema for production).

### DB schema
```
annotations(id, claim, evidence, label, confidence, comment,
            annotator, ticker, filing_type, filing_date,
            section_title, chunk_id, timestamp)
```

### Expected outputs
- `annotations/annotations.db` (SQLite)
- `annotations/annotations_export.parquet`

In [83]:
from sqlalchemy import (
    Column, Integer, Float, String, Text, DateTime, create_engine)
from sqlalchemy.orm import DeclarativeBase, Session
from sqlalchemy.sql import func
import gradio as gr


class _Base(DeclarativeBase): pass

class AnnotationRecord(_Base):
    __tablename__ = 'annotations'
    id            = Column(Integer, primary_key=True, autoincrement=True)
    claim         = Column(Text, nullable=False)
    evidence      = Column(Text, nullable=False)
    label         = Column(String(20), nullable=False)
    confidence    = Column(Float, nullable=True)
    comment       = Column(Text, nullable=True)
    annotator     = Column(String(100), nullable=True)
    ticker        = Column(String(10), nullable=True)
    filing_type   = Column(String(10), nullable=True)
    filing_date   = Column(String(20), nullable=True)
    section_title = Column(Text, nullable=True)
    chunk_id      = Column(String(64), nullable=True)
    timestamp     = Column(DateTime, server_default=func.now())


class AnnotationStore:
    """SQLAlchemy-backed annotation store. Swap URI for PostgreSQL in production."""

    VALID_LABELS = {'ENTAILMENT','CONTRADICTION','NEUTRAL'}

    def __init__(self, db_url: Optional[str] = None):
        url = db_url or f'sqlite:///{cfg.ANNOTATIONS_DIR / "annotations.db"}'
        if url.startswith('sqlite:///') and not url.startswith('sqlite://///'):
            rel = url[len('sqlite:///'):]  
            abs_p = cfg.ANNOTATIONS_DIR / rel if not Path(rel).is_absolute() else Path(rel)
            abs_p.parent.mkdir(parents=True, exist_ok=True)
            url = f'sqlite:///{abs_p}'
        self.engine = create_engine(url, echo=False, future=True)
        _Base.metadata.create_all(self.engine)
        log.info('AnnotationStore ready — %s', url)

    def save(self, claim, evidence, label, confidence=None, comment=None,
             annotator=None, metadata=None) -> int:
        label = label.upper().strip()
        if label not in self.VALID_LABELS:
            raise ValueError(f'label must be one of {self.VALID_LABELS}')
        meta = metadata or {}
        rec = AnnotationRecord(
            claim=claim, evidence=evidence, label=label, confidence=confidence,
            comment=comment, annotator=annotator or 'anonymous',
            ticker=meta.get('ticker'), filing_type=meta.get('filing_type'),
            filing_date=meta.get('filing_date'), section_title=meta.get('section_title'),
            chunk_id=meta.get('chunk_id'))
        with Session(self.engine) as s:
            s.add(rec); s.commit(); s.refresh(rec); return rec.id

    def load_all(self) -> pd.DataFrame:
        with Session(self.engine) as s:
            recs = s.query(AnnotationRecord).all()
        if not recs: return pd.DataFrame()
        return pd.DataFrame([{
            'id':r.id,'claim':r.claim,'evidence':r.evidence,'label':r.label,
            'confidence':r.confidence,'comment':r.comment,'annotator':r.annotator,
            'ticker':r.ticker,'filing_type':r.filing_type,'filing_date':r.filing_date,
            'section_title':r.section_title,'chunk_id':r.chunk_id,'timestamp':str(r.timestamp)
        } for r in recs])

    def export_parquet(self, path: Optional[Path] = None) -> Path:
        out = path or (cfg.ANNOTATIONS_DIR / 'annotations_export.parquet')
        df = self.load_all()
        df.to_parquet(out, index=False)
        log.info('Annotations exported → %s (%d rows)', out, len(df))
        return out

    def count(self) -> Dict[str, int]:
        df = self.load_all()
        return df['label'].value_counts().to_dict() if not df.empty else {}


annotation_store = AnnotationStore()
log.info('AnnotationStore instantiated')


2026-07-02 19:03:58 | INFO     | hallucination_engine | AnnotationStore ready — sqlite:////home/lonewolf/quant/Projects/FINANCIAL_LLM/v3/data/annotations/annotations.db
2026-07-02 19:03:58 | INFO     | hallucination_engine | AnnotationStore instantiated


In [84]:
# Seed synthetic annotations
SEED_ROWS = [
    ('Apple iPhone revenue was approximately $200.6 billion in fiscal 2023.',
     'Revenue from iPhone was $200.6 billion, representing 52 percent of total net sales.',
     'ENTAILMENT','AAPL','10-K','2023-11-03','Item 7. MD&A'),
    ('Microsoft total revenue grew 7 percent to $211.9 billion in fiscal 2023.',
     'Revenue for fiscal year 2023 was $211.9 billion, representing 7 percent growth.',
     'ENTAILMENT','MSFT','10-K','2023-07-27','Item 1. Business'),
    ('Apple Services revenue hit a record $85.2 billion.',
     'Services revenue reached a record $85.2 billion, growing 9 percent year-over-year.',
     'ENTAILMENT','AAPL','10-K','2023-11-03','Item 7. MD&A'),
    ('Microsoft Azure grew 29 percent in fiscal 2023.',
     'Azure and other cloud services revenue grew 29 percent.',
     'ENTAILMENT','MSFT','10-K','2023-07-27','Item 7. MD&A'),
    ('Apple net sales declined 2.8 percent in fiscal 2023.',
     'Net sales for fiscal 2023 were $383.3 billion, a decrease of 2.8 percent.',
     'ENTAILMENT','AAPL','10-K','2023-11-03','Item 1. Business'),
    ('Apple total net sales grew 5 percent in fiscal 2023.',
     'Net sales for fiscal 2023 were $383.3 billion, a decrease of 2.8 percent.',
     'CONTRADICTION','AAPL','10-K','2023-11-03','Item 1. Business'),
    ('Microsoft net income was $50 billion in fiscal 2023.',
     'Net income was $72.4 billion.',
     'CONTRADICTION','MSFT','10-K','2023-07-27','Item 7. MD&A'),
    ('Apple iPhone revenue declined by 10 percent in fiscal 2023.',
     'Revenue from iPhone was $200.6 billion, representing 52 percent of total net sales.',
     'CONTRADICTION','AAPL','10-K','2023-11-03','Item 7. MD&A'),
    ('Microsoft Azure grew only 10 percent in fiscal 2023.',
     'Azure and other cloud services revenue grew 29 percent.',
     'CONTRADICTION','MSFT','10-K','2023-07-27','Item 7. MD&A'),
    ('Apple faces no competition in its markets.',
     'The markets for the Company\'s products are highly competitive.',
     'CONTRADICTION','AAPL','10-K','2023-11-03','Item 1A. Risk Factors'),
    ('Apple plans to release an AR headset next quarter.',
     'Net sales for fiscal 2023 were $383.3 billion, a decrease of 2.8 percent.',
     'NEUTRAL','AAPL','10-K','2023-11-03','Item 1. Business'),
    ('Microsoft has offices in over 100 countries.',
     'Azure and other cloud services revenue grew 29 percent.',
     'NEUTRAL','MSFT','10-K','2023-07-27','Item 7. MD&A'),
    ('Apple employs over 150,000 people worldwide.',
     'The Company faces substantial competition in all areas of its business.',
     'NEUTRAL','AAPL','10-K','2023-11-03','Item 1A. Risk Factors'),
    ('Microsoft was founded by Bill Gates and Paul Allen.',
     'Microsoft Cloud revenue grew 22 percent to $111.6 billion.',
     'NEUTRAL','MSFT','10-K','2023-07-27','Item 7. MD&A'),
    ('Apple stock has outperformed the S&P 500 over the past decade.',
     'Apple Inc. designs, manufactures, and markets smartphones and personal computers.',
     'NEUTRAL','AAPL','10-K','2023-11-03','Item 1. Business'),
]

if annotation_store.count().get('ENTAILMENT', 0) == 0:
    for claim, evidence, label, ticker, form, date, section in SEED_ROWS:
        annotation_store.save(claim=claim, evidence=evidence, label=label,
            annotator='seed_script',
            metadata={'ticker':ticker,'filing_type':form,'filing_date':date,'section_title':section})
    log.info('Seeded %d annotations', len(SEED_ROWS))

print('Annotation label counts:', annotation_store.count())


Annotation label counts: {'ENTAILMENT': 5, 'CONTRADICTION': 5, 'NEUTRAL': 5}


In [85]:
# ── Gradio annotation UI ──────────────────────────────────────────────────────
_last_results: List[RetrievalResult] = []

def _do_retrieve(claim: str, ticker: str, top_k: int) -> str:
    global _last_results
    if not claim.strip(): return 'Enter a claim first.'
    try:
        _last_results = retrieval_pipeline.retrieve(
            claim, top_k=int(top_k),
            ticker_filter=ticker.strip().upper() or None)
        return retrieval_pipeline.format_evidence(_last_results)
    except Exception as exc:
        return f'Error: {exc}'

def _do_submit(claim, evidence_override, label, confidence, comment, annotator) -> str:
    if not claim.strip(): return '⚠️ Enter a claim.'
    if not label: return '⚠️ Select a label.'
    ev = evidence_override.strip() or (_last_results[0].text if _last_results else '')
    meta = {}
    if _last_results:
        r = _last_results[0]
        meta = {'ticker':r.ticker,'filing_type':r.filing_type,
                'filing_date':r.filing_date,'section_title':r.section_title,'chunk_id':r.chunk_id}
    try:
        rid = annotation_store.save(claim=claim, evidence=ev, label=label,
            confidence=float(confidence), comment=comment or None,
            annotator=annotator or 'anonymous', metadata=meta)
        c = annotation_store.count()
        return (f' Saved #{rid} | E={c.get("ENTAILMENT",0)} '
                f'C={c.get("CONTRADICTION",0)} N={c.get("NEUTRAL",0)}')
    except Exception as exc:
        return f' {exc}'

def build_annotation_app() -> gr.Blocks:
    with gr.Blocks(title='TR Hallucination Annotation', theme=gr.themes.Soft()) as app:
        gr.Markdown('#  SEC Hallucination Annotation Tool\n'
            '**Entailment** · **Contradiction** · **Neutral**')
        with gr.Row():
            with gr.Column(scale=1):
                claim_box = gr.Textbox(label='LLM Claim', lines=4)
                with gr.Row():
                    ticker_box  = gr.Textbox(label='Ticker filter', placeholder='AAPL', scale=1)
                    topk_slider = gr.Slider(1, 10, value=cfg.TOP_K, step=1, label='Top-K', scale=2)
                retrieve_btn = gr.Button(' Retrieve Evidence', variant='primary')
            with gr.Column(scale=2):
                evidence_box = gr.Textbox(label='Retrieved Evidence', lines=12, interactive=False)
        with gr.Row():
            label_radio  = gr.Radio(['ENTAILMENT','CONTRADICTION','NEUTRAL'], label='NLI Label', scale=2)
            conf_slider  = gr.Slider(1, 5, value=3, step=1, label='Confidence', scale=1)
        with gr.Row():
            ev_override = gr.Textbox(label='Evidence override (optional)', lines=2, scale=3)
            annotator   = gr.Textbox(label='Annotator ID', placeholder='your.name', scale=1)
        comment_box  = gr.Textbox(label='Comment (optional)', lines=2)
        submit_btn   = gr.Button(' Submit', variant='secondary')
        status_text  = gr.Textbox(label='Status', interactive=False)
        retrieve_btn.click(_do_retrieve, [claim_box, ticker_box, topk_slider], [evidence_box])
        submit_btn.click(_do_submit,
            [claim_box, ev_override, label_radio, conf_slider, comment_box, annotator],
            [status_text])
    return app

annotation_app = build_annotation_app()

LAUNCH_GRADIO: bool = False   # set True for interactive annotation
if LAUNCH_GRADIO:
    annotation_app.launch(server_name='0.0.0.0', server_port=7860, share=False, quiet=True)
else:
    log.info('Gradio app built (LAUNCH_GRADIO=False). Set True to open UI.')

annotations_export_path = annotation_store.export_parquet()
annotations_df = pd.read_parquet(annotations_export_path)
print(f'Annotations available: {len(annotations_df)}')


2026-07-02 19:03:58 | INFO     | hallucination_engine | Gradio app built (LAUNCH_GRADIO=False). Set True to open UI.
2026-07-02 19:03:58 | INFO     | hallucination_engine | Annotations exported → /home/lonewolf/quant/Projects/FINANCIAL_LLM/v3/data/annotations/annotations_export.parquet (15 rows)
Annotations available: 15


In [86]:
from sklearn.model_selection import train_test_split

LABEL2ID: Dict[str, int] = {'CONTRADICTION': 0, 'ENTAILMENT': 1, 'NEUTRAL': 2}
ID2LABEL: Dict[int, str] = {v: k for k, v in LABEL2ID.items()}
NUM_LABELS: int = len(LABEL2ID)


class NLIDatasetBuilder:
    """Convert annotation store records into stratified NLI CSV splits.

    Output schema: premise, hypothesis, label (int), label_str, chunk_id, ticker.
    """

    _MIN_CHARS = 10

    def __init__(self, config: PipelineConfig): self.cfg = config

    def build(self, annotation_df: Optional[pd.DataFrame] = None,
              annotation_path: Optional[Path] = None) -> Dict[str, pd.DataFrame]:
        if annotation_df is None:
            p = annotation_path or (self.cfg.ANNOTATIONS_DIR / 'annotations_export.parquet')
            annotation_df = pd.read_parquet(p)
        nli = self._transform(annotation_df)
        splits = self._split(nli)
        self._save(splits); self._report(splits)
        return splits

    def _transform(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        df['label_str'] = df['label'].str.upper().str.strip()
        df = df[df['label_str'].isin(LABEL2ID)]
        df = df[(df['claim'].str.len() >= self._MIN_CHARS) &
                (df['evidence'].str.len() >= self._MIN_CHARS)]
        nli = pd.DataFrame({
            'premise':    df['evidence'].str.strip(),
            'hypothesis': df['claim'].str.strip(),
            'label':      df['label_str'].map(LABEL2ID).astype(int),
            'label_str':  df['label_str'],
            'chunk_id':   df.get('chunk_id', pd.Series([''] * len(df))).values,
            'ticker':     df.get('ticker',   pd.Series([''] * len(df))).values,
        }).reset_index(drop=True)
        pre = len(nli)
        nli = nli.drop_duplicates(subset=['premise','hypothesis'])
        log.info('NLI triples: %d (deduped %d) | dist=%s',
                 len(nli), pre-len(nli), nli['label_str'].value_counts().to_dict())
        return nli

    def _split(self, df: pd.DataFrame) -> Dict[str, pd.DataFrame]:
        labels = df['label']
        stratify = labels if labels.value_counts().min() >= 3 else None
        if stratify is None:
            log.warning('Too few samples per class — using random split')
        val_test = self.cfg.VAL_RATIO + self.cfg.TEST_RATIO
        train, valtest = train_test_split(df, test_size=val_test, random_state=SEED,
                                         stratify=stratify)
        vt_strat = None

        if valtest["label"].value_counts().min() >= 2:
            vt_strat = valtest["label"]

        val, test = train_test_split(
            valtest,
            test_size=0.5,
            random_state=SEED,
            stratify=vt_strat,
        )
        return {'train': train.reset_index(drop=True),
                'validation': val.reset_index(drop=True),
                'test': test.reset_index(drop=True)}

    def _save(self, splits):
        for name, df in splits.items():
            p = self.cfg.ANNOTATIONS_DIR / f'{name}.csv'
            df.to_csv(p, index=False)
            log.info('%s → %s (%d rows)', name, p, len(df))

    @staticmethod
    def _report(splits):
        for name, df in splits.items():
            print(f'{name:12s} | n={len(df):4d} | {df["label_str"].value_counts().to_dict()}')


dataset_builder = NLIDatasetBuilder(cfg)
nli_splits = dataset_builder.build(annotation_df=annotations_df)
train_df = nli_splits['train']
val_df   = nli_splits['validation']
test_df  = nli_splits['test']
print(f'\nLabel mapping: {LABEL2ID}')
print(f'Sample: {train_df[["premise","hypothesis","label_str"]].iloc[0].to_dict()}')


2026-07-02 19:03:58 | INFO     | hallucination_engine | NLI triples: 15 (deduped 0) | dist={'ENTAILMENT': 5, 'CONTRADICTION': 5, 'NEUTRAL': 5}
2026-07-02 19:03:58 | INFO     | hallucination_engine | train → /home/lonewolf/quant/Projects/FINANCIAL_LLM/v3/data/annotations/train.csv (10 rows)
2026-07-02 19:03:58 | INFO     | hallucination_engine | validation → /home/lonewolf/quant/Projects/FINANCIAL_LLM/v3/data/annotations/validation.csv (2 rows)
2026-07-02 19:03:58 | INFO     | hallucination_engine | test → /home/lonewolf/quant/Projects/FINANCIAL_LLM/v3/data/annotations/test.csv (3 rows)
train        | n=  10 | {'CONTRADICTION': 4, 'ENTAILMENT': 3, 'NEUTRAL': 3}
validation   | n=   2 | {'NEUTRAL': 2}
test         | n=   3 | {'ENTAILMENT': 2, 'CONTRADICTION': 1}

Label mapping: {'CONTRADICTION': 0, 'ENTAILMENT': 1, 'NEUTRAL': 2}
Sample: {'premise': 'Net sales for fiscal 2023 were $383.3 billion, a decrease of 2.8 percent.', 'hypothesis': 'Apple total net sales grew 5 percent in fiscal 202

In [87]:
from transformers import (
    AutoModelForSequenceClassification, AutoTokenizer as _ATok,
    DataCollatorWithPadding, EarlyStoppingCallback,
    Trainer, TrainingArguments,
)
from datasets import Dataset
import evaluate
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, classification_report)

_acc_metric = evaluate.load('accuracy')


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy':        float(accuracy_score(labels, preds)),
        'f1_macro':        float(f1_score(labels, preds, average='macro',    zero_division=0)),
        'f1_weighted':     float(f1_score(labels, preds, average='weighted', zero_division=0)),
        'precision_macro': float(precision_score(labels, preds, average='macro', zero_division=0)),
        'recall_macro':    float(recall_score(labels,  preds, average='macro', zero_division=0)),
    }


class DeBERTaTrainer:
    """Fine-tune DeBERTa-v3-Large on 3-class NLI (E/N/C)."""

    def __init__(self, config: PipelineConfig):
        self.cfg = config
        self.model = None; self.tokenizer = None; self.trainer = None

    def _tokenise(self, df: pd.DataFrame) -> Dataset:
        ds = Dataset.from_pandas(df[['premise','hypothesis','label']].reset_index(drop=True))
        ds = ds.map(lambda b: self.tokenizer(
            b['premise'], b['hypothesis'],
            truncation=True, padding=False,
            max_length=self.cfg.MAX_SEQ_LENGTH),
            batched=True, remove_columns=['premise','hypothesis'], desc='Tokenising')
        ds = ds.rename_column('label','labels')
        ds.set_format('torch')
        return ds

    def train(self, train_df: pd.DataFrame, val_df: pd.DataFrame) -> 'DeBERTaTrainer':
        log.info('=== DeBERTa fine-tuning | train=%d val=%d ===', len(train_df), len(val_df))

        log.info('Loading tokenizer: %s', self.cfg.NLI_MODEL)
        self.tokenizer = _ATok.from_pretrained(self.cfg.NLI_MODEL)
        train_ds = self._tokenise(train_df)
        val_ds   = self._tokenise(val_df)

        log.info('Loading model: %s', self.cfg.NLI_MODEL)
        self.model = AutoModelForSequenceClassification.from_pretrained(
            self.cfg.NLI_MODEL,
            num_labels=NUM_LABELS,
            id2label=ID2LABEL,
            label2id=LABEL2ID,
            # ignore_mismatched_sizes must be False: the base NLI model
            # already has a 3-class head trained on NLI data.  Enabling
            # this flag would reinitialise that head with random weights,
            # destroying the pretrained NLI signal.
            ignore_mismatched_sizes=False,
        ).to(DEVICE)

        total_steps = (len(train_ds) // max(1,
            self.cfg.BATCH_SIZE * self.cfg.GRADIENT_ACCUMULATION_STEPS)) * self.cfg.EPOCHS
        warmup = max(1, int(total_steps * self.cfg.WARMUP_RATIO))

        args = TrainingArguments(
            output_dir=str(self.cfg.MODEL_OUTPUT_DIR),
            num_train_epochs=self.cfg.EPOCHS,
            per_device_train_batch_size=self.cfg.BATCH_SIZE,
            per_device_eval_batch_size=self.cfg.BATCH_SIZE * 2,
            gradient_accumulation_steps=self.cfg.GRADIENT_ACCUMULATION_STEPS,
            learning_rate=self.cfg.LEARNING_RATE,
            weight_decay=self.cfg.WEIGHT_DECAY,
            warmup_steps=warmup,
            lr_scheduler_type='linear',
            evaluation_strategy='epoch',
            save_strategy='epoch',
            logging_strategy='steps',
            logging_steps=max(1, len(train_ds) // max(1, self.cfg.BATCH_SIZE * 10)),
            load_best_model_at_end=True,
            metric_for_best_model='f1_macro',
            greater_is_better=True,
            fp16=self.cfg.FP16, bf16=False,
            dataloader_num_workers=0,
            report_to='none',
            seed=SEED,
            save_total_limit=2,
        )
        log.info('Effective batch=%d | warmup=%d | fp16=%s',
                 self.cfg.BATCH_SIZE * self.cfg.GRADIENT_ACCUMULATION_STEPS,
                 warmup, self.cfg.FP16)

        self.trainer = Trainer(
            model=self.model, args=args,
            train_dataset=train_ds, eval_dataset=val_ds,
            tokenizer=self.tokenizer,
            data_collator=DataCollatorWithPadding(
                self.tokenizer, pad_to_multiple_of=8 if self.cfg.FP16 else None),
            compute_metrics=compute_metrics,
            callbacks=[EarlyStoppingCallback(
                early_stopping_patience=2, early_stopping_threshold=0.001)],
        )

        result = self.trainer.train()
        log.info('Training complete — %s', result.metrics)
        self._save()
        return self

    def _save(self):
        self.trainer.save_model(str(self.cfg.MODEL_OUTPUT_DIR))
        self.tokenizer.save_pretrained(str(self.cfg.MODEL_OUTPUT_DIR))
        lp = self.cfg.MODEL_OUTPUT_DIR / 'label_map.json'
        lp.write_text(json.dumps({'id2label': ID2LABEL, 'label2id': LABEL2ID}, indent=2))
        log.info('Model saved → %s', self.cfg.MODEL_OUTPUT_DIR)

    @classmethod
    def load_trained(cls, config: PipelineConfig) -> 'DeBERTaTrainer':
        inst = cls(config)
        inst.tokenizer = _ATok.from_pretrained(str(config.MODEL_OUTPUT_DIR))
        # Load exactly as saved. The checkpoint's config.json already
        # contains the correct num_labels/id2label/label2id from training
        # (see ._save()); overriding them here — or setting
        # ignore_mismatched_sizes=True — would either clobber the saved
        # label mapping with a stale global dict or reinitialise the
        # classification head with random weights.
        inst.model = AutoModelForSequenceClassification.from_pretrained(
            str(config.MODEL_OUTPUT_DIR)
        ).to(DEVICE).eval()
        log.info('Loaded fine-tuned model from %s', config.MODEL_OUTPUT_DIR)
        return inst


log.info('DeBERTaTrainer defined')


IMPORTANT: You are using gradio version 4.36.1, however version 4.44.1 is available, please upgrade.
--------
2026-07-02 19:03:59 | INFO     | hallucination_engine | DeBERTaTrainer defined


In [88]:
SKIP_TRAINING: bool = False
_model_exists = (cfg.MODEL_OUTPUT_DIR / 'config.json').exists()

nli_trainer = DeBERTaTrainer(cfg)

if SKIP_TRAINING and _model_exists:
    nli_trainer = DeBERTaTrainer.load_trained(cfg)
else:
    if len(train_df) < 3:
        log.warning('Only %d training samples — expand annotation corpus for production.', len(train_df))
    nli_trainer.train(train_df=train_df, val_df=val_df)

log.info('Section 11 complete — model at %s', cfg.MODEL_OUTPUT_DIR)

2026-07-02 19:04:00 | INFO     | hallucination_engine | === DeBERTa fine-tuning | train=10 val=2 ===
2026-07-02 19:04:00 | INFO     | hallucination_engine | Loading tokenizer: cross-encoder/nli-deberta-v3-small


Tokenising:   0%|          | 0/10 [00:00<?, ? examples/s]

Tokenising:   0%|          | 0/2 [00:00<?, ? examples/s]

2026-07-02 19:04:27 | INFO     | hallucination_engine | Loading model: cross-encoder/nli-deberta-v3-small
2026-07-02 19:04:28 | INFO     | hallucination_engine | Effective batch=32 | warmup=1 | fp16=True


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted,Precision Macro,Recall Macro
1,0.794900,0.000668,1.000000,1.000000,1.000000,1.000000,1.000000
2,1.198900,0.000756,1.000000,1.000000,1.000000,1.000000,1.000000
3,0.951700,0.000817,1.000000,1.000000,1.000000,1.000000,1.000000


2026-07-02 19:04:35 | INFO     | hallucination_engine | Training complete — {'train_runtime': 6.2561, 'train_samples_per_second': 7.992, 'train_steps_per_second': 0.799, 'total_flos': 302207406624.0, 'train_loss': 0.9818567037582397, 'epoch': 3.0}
2026-07-02 19:04:35 | INFO     | hallucination_engine | Model saved → /home/lonewolf/quant/Projects/FINANCIAL_LLM/v3/models/fine_tuned_model
2026-07-02 19:04:35 | INFO     | hallucination_engine | Section 11 complete — model at /home/lonewolf/quant/Projects/FINANCIAL_LLM/v3/models/fine_tuned_model


In [89]:
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

if nli_trainer.trainer is not None:
    hist = nli_trainer.trainer.state.log_history
    t_steps, t_loss, e_epochs, e_f1, e_acc = [], [], [], [], []
    for h in hist:
        if 'loss' in h: t_steps.append(h.get('step',0)); t_loss.append(h['loss'])
        if 'eval_f1_macro' in h:
            e_epochs.append(h.get('epoch',0))
            e_f1.append(h['eval_f1_macro'])
            e_acc.append(h.get('eval_accuracy',0))

    if t_steps:
        fig, ax = plt.subplots(1, 2, figsize=(12,4))
        ax[0].plot(t_steps, t_loss, 'b-o', ms=4, label='train loss')
        ax[0].set_title('Training Loss'); ax[0].set_xlabel('Step'); ax[0].legend(); ax[0].grid(alpha=0.3)
        if e_epochs:
            ax[1].plot(e_epochs, e_f1,  'g-s', ms=6, label='val F1 macro')
            ax[1].plot(e_epochs, e_acc, 'r-^', ms=6, label='val Accuracy')
            ax[1].set_title('Validation Metrics'); ax[1].set_xlabel('Epoch')
            ax[1].legend(); ax[1].grid(alpha=0.3)
        plt.tight_layout()
        out = cfg.OUTPUTS_DIR / 'training_curves.png'
        plt.savefig(out, dpi=120, bbox_inches='tight'); plt.show()
        log.info('Training curves → %s', out)

2026-07-02 19:04:36 | INFO     | hallucination_engine | Training curves → /home/lonewolf/quant/Projects/FINANCIAL_LLM/v3/outputs/training_curves.png


In [90]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer as _AutoTok2


def load_nli_model(model_dir: Path):
    """Load the fine-tuned NLI model and tokenizer.

    Falls back to the pretrained cross-encoder when no fine-tuned checkpoint
    exists.  In both paths the model's id2label is normalised to uppercase
    so that HallucinationDetector can decode predictions consistently via
    model.config.id2label without relying on the global ID2LABEL dict.
    """
    if (model_dir / 'config.json').exists():
        log.info('Loading fine-tuned model from %s', model_dir)
        # Load exactly as saved — do NOT pass ignore_mismatched_sizes here;
        # the serialised model already has the correct 3-class head.
        model = AutoModelForSequenceClassification.from_pretrained(
            str(model_dir)
        )
        tokenizer = _AutoTok2.from_pretrained(str(model_dir))
    else:
        log.warning(
            'Fine-tuned model not found at %s — loading pretrained NLI fallback.',
            model_dir
        )
        model = AutoModelForSequenceClassification.from_pretrained(
            cfg.NLI_MODEL   # cross-encoder/nli-deberta-v3-small
        )
        tokenizer = _AutoTok2.from_pretrained(cfg.NLI_MODEL)

    # HallucinationDetector reads model.config.id2label
    # directly — this is the single source of truth for label decoding.
    model.config.id2label  = {k: v.upper() for k, v in model.config.id2label.items()}
    model.config.label2id  = {v: k for k, v in model.config.id2label.items()}

    return model.to(DEVICE).eval(), tokenizer


nli_model, nli_tokenizer = load_nli_model(cfg.MODEL_OUTPUT_DIR)

# Keep nli_trainer attribute references alive for any cells that use them
if hasattr(nli_trainer, 'model'):
    nli_trainer.model     = nli_model
    nli_trainer.tokenizer = nli_tokenizer

log.info(
    'NLI model ready | id2label=%s | device=%s',
    nli_model.config.id2label, DEVICE
)


2026-07-02 19:04:36 | INFO     | hallucination_engine | Loading fine-tuned model from /home/lonewolf/quant/Projects/FINANCIAL_LLM/v3/models/fine_tuned_model
2026-07-02 19:04:36 | INFO     | hallucination_engine | NLI model ready | id2label={0: 'CONTRADICTION', 1: 'ENTAILMENT', 2: 'NEUTRAL'} | device=cuda


In [91]:
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
)
from sklearn.preprocessing import label_binarize
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns


class ModelEvaluator:
    """Full evaluation suite for the NLI hallucination detector.

    Runs batch inference on a test DataFrame, then produces every metric
    required for a production model card.

    Parameters
    ----------
    model:      Fine-tuned AutoModelForSequenceClassification.
    tokenizer:  Matching tokenizer.
    config:     Global PipelineConfig.
    """

    def __init__(self, model, tokenizer, config: PipelineConfig) -> None:
        self.model     = model
        self.tokenizer = tokenizer
        self.cfg       = config
        self.model.eval()

    def evaluate(
        self,
        test_df: pd.DataFrame,
        batch_size: int = 16,
    ) -> Dict[str, Any]:
        """Run full evaluation and return metrics dict.

        Parameters
        ----------
        test_df:    DataFrame with columns premise, hypothesis, label.
        batch_size: Inference batch size.

        Returns
        -------
        Dict containing all scalar metrics, classification report,
        confusion matrix, ROC data, and error examples.
        """
        if test_df.empty:
            log.warning('test_df is empty — skipping evaluation.')
            return {}

        log.info('Running inference on %d test samples...', len(test_df))
        preds, probs, true_labels = self._predict(test_df, batch_size)

        metrics = self._compute_metrics(true_labels, preds, probs)
        metrics['error_examples'] = self._error_examples(test_df, preds, true_labels)

        self._save_report(metrics)
        self._plot_confusion_matrix(true_labels, preds)
        self._plot_roc_curves(true_labels, probs)
        self._save_error_analysis(metrics['error_examples'])

        log.info(
            'Evaluation complete — Acc=%.4f | F1-macro=%.4f | ROC-AUC=%.4f',
            metrics['accuracy'], metrics['f1_macro'],
            metrics.get('roc_auc_macro', float('nan'))
        )
        return metrics

    @torch.no_grad()
    def _predict(
        self,
        df: pd.DataFrame,
        batch_size: int,
    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        """Return (predictions, softmax_probs, true_labels) arrays."""
        all_preds, all_probs, all_labels = [], [], []

        for start in tqdm(range(0, len(df), batch_size), desc='Evaluating'):
            batch = df.iloc[start:start + batch_size]
            premises    = batch['premise'].tolist()
            hypotheses  = batch['hypothesis'].tolist()
            true_batch  = batch['label'].tolist()

            enc = self.tokenizer(
                premises, hypotheses,
                truncation=True,
                padding=True,
                max_length=self.cfg.MAX_SEQ_LENGTH,
                return_tensors='pt',
            ).to(DEVICE)

            logits = self.model(**enc).logits          # (B, num_labels)
            probs  = torch.softmax(logits, dim=-1).cpu().numpy()
            preds  = np.argmax(probs, axis=-1)

            all_preds.extend(preds.tolist())
            all_probs.extend(probs.tolist())
            all_labels.extend(true_batch)

        return (
            np.array(all_preds, dtype=int),
            np.array(all_probs, dtype=np.float32),
            np.array(all_labels, dtype=int),
        )

    # metrics

    @staticmethod
    def _compute_metrics(
        true_labels: np.ndarray,
        preds: np.ndarray,
        probs: np.ndarray,
    ) -> Dict[str, Any]:
        label_names = [ID2LABEL[i] for i in range(NUM_LABELS)]

        acc      = float(accuracy_score(true_labels, preds))
        f1_mac   = float(f1_score(true_labels, preds, average='macro',    zero_division=0))
        f1_wei   = float(f1_score(true_labels, preds, average='weighted', zero_division=0))
        prec_mac = float(precision_score(true_labels, preds, average='macro',    zero_division=0))
        rec_mac  = float(recall_score(true_labels,  preds, average='macro',    zero_division=0))

        clf_report = classification_report(
            true_labels, preds,
            target_names=label_names,
            output_dict=True,
            zero_division=0,
        )
        cm = confusion_matrix(true_labels, preds).tolist()

        # ROC-AUC (one-vs-rest, macro)
        roc_auc_macro = float('nan')
        roc_data: Dict[str, Dict] = {}
        try:
            y_bin = label_binarize(true_labels, classes=list(range(NUM_LABELS)))
            roc_auc_macro = float(roc_auc_score(y_bin, probs, multi_class='ovr', average='macro'))
            for i, name in enumerate(label_names):
                fpr, tpr, _ = roc_curve(y_bin[:, i], probs[:, i])
                roc_data[name] = {'fpr': fpr.tolist(), 'tpr': tpr.tolist()}
        except Exception as exc:
            log.warning('ROC-AUC computation failed: %s', exc)

        return {
            'accuracy':              acc,
            'f1_macro':              f1_mac,
            'f1_weighted':           f1_wei,
            'precision_macro':       prec_mac,
            'recall_macro':          rec_mac,
            'roc_auc_macro':         roc_auc_macro,
            'classification_report': clf_report,
            'confusion_matrix':      cm,
            'roc_data':              roc_data,
            'predictions':           preds.tolist(),
            'true_labels':           true_labels.tolist(),
            'softmax_probs':         probs.tolist(),
        }

    # error analysis

    @staticmethod
    def _error_examples(
        df: pd.DataFrame,
        preds: np.ndarray,
        true_labels: np.ndarray,
        n: int = 10,
    ) -> List[Dict[str, str]]:
        """Return up to *n* misclassified examples with predicted/true labels."""
        errors = []
        for i, (pred, true) in enumerate(zip(preds, true_labels)):
            if pred != true:
                row = df.iloc[i]
                errors.append({
                    'index':          i,
                    'premise':        str(row.get('premise',  ''))[:300],
                    'hypothesis':     str(row.get('hypothesis',''))[:200],
                    'true_label':     ID2LABEL[int(true)],
                    'predicted_label':ID2LABEL[int(pred)],
                    'ticker':         str(row.get('ticker', '')),
                })
            if len(errors) == n:
                break
        return errors

    # plots 

    def _plot_confusion_matrix(self, true_labels, preds) -> None:
        label_names = [ID2LABEL[i] for i in range(NUM_LABELS)]
        cm = confusion_matrix(true_labels, preds)

        fig, axes = plt.subplots(1, 2, figsize=(12, 5))

        # Raw counts
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=label_names, yticklabels=label_names, ax=axes[0])
        axes[0].set_title('Confusion Matrix (counts)')
        axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')

        # Normalised
        cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-9)
        sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                    xticklabels=label_names, yticklabels=label_names, ax=axes[1])
        axes[1].set_title('Confusion Matrix (normalised)')
        axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')

        plt.tight_layout()
        out = self.cfg.OUTPUTS_DIR / 'confusion_matrix.png'
        plt.savefig(out, dpi=120, bbox_inches='tight')
        plt.show()
        log.info('Confusion matrix saved → %s', out)

    def _plot_roc_curves(self, true_labels, probs) -> None:
        label_names = [ID2LABEL[i] for i in range(NUM_LABELS)]
        y_bin = label_binarize(true_labels, classes=list(range(NUM_LABELS)))

        # Need at least 2 classes present for ROC
        if y_bin.shape[1] < 2 or y_bin.sum(axis=0).min() == 0:
            log.warning('Insufficient class diversity for ROC curves — skipping.')
            return

        colors = ['steelblue', 'darkorange', 'forestgreen']
        fig, ax = plt.subplots(figsize=(8, 6))

        for i, (name, color) in enumerate(zip(label_names, colors)):
            try:
                fpr, tpr, _ = roc_curve(y_bin[:, i], probs[:, i])
                auc = roc_auc_score(y_bin[:, i], probs[:, i])
                ax.plot(fpr, tpr, color=color, lw=2,
                        label=f'{name} (AUC={auc:.3f})')
            except Exception:
                pass

        ax.plot([0,1], [0,1], 'k--', lw=1, label='Random')
        ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
        ax.set_title('ROC Curves – One-vs-Rest (NLI labels)')
        ax.legend(loc='lower right'); ax.grid(alpha=0.3)

        out = self.cfg.OUTPUTS_DIR / 'roc_curves.png'
        plt.savefig(out, dpi=120, bbox_inches='tight')
        plt.show()
        log.info('ROC curves saved → %s', out)

    # persistence

    def _save_report(self, metrics: Dict[str, Any]) -> None:
        report = {k: v for k, v in metrics.items()
                  if k not in {'predictions','true_labels','softmax_probs',
                               'roc_data','error_examples'}}
        out = self.cfg.OUTPUTS_DIR / 'evaluation_report.json'
        out.write_text(json.dumps(report, indent=2, default=str), encoding='utf-8')
        log.info('Evaluation report saved → %s', out)

    def _save_error_analysis(self, errors: List[Dict]) -> None:
        if not errors:
            return
        out = self.cfg.OUTPUTS_DIR / 'error_analysis.csv'
        pd.DataFrame(errors).to_csv(out, index=False)
        log.info('Error analysis saved → %s  (%d errors)', out, len(errors))


log.info('ModelEvaluator class defined')


2026-07-02 19:04:36 | INFO     | hallucination_engine | ModelEvaluator class defined


In [92]:
@dataclass
class ClaimVerdict:
    """Complete verification result for a single LLM-generated claim."""
    claim_index:       int
    claim:             str
    evidence_text:     str
    evidence_ticker:   str
    evidence_section:  str
    evidence_date:     str
    evidence_page:     int
    prediction:        str     # ENTAILMENT | NEUTRAL | CONTRADICTION
    confidence:        float   # softmax probability of predicted class
    prob_entailment:   float
    prob_neutral:      float
    prob_contradiction:float
    hallucination_flag:bool    # True = claim contradicted or unverifiable
    retrieval_score:   float   # cosine similarity of top evidence

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


class HallucinationDetector:
    """End-to-end hallucination detection for financial LLM outputs.

    Combines:
    - Rule-based claim splitter (sentence boundaries)
    - Vector retrieval (Section 7 / 8)
    - DeBERTa NLI classifier (Section 11)

    Parameters
    ----------
    model:              Fine-tuned DeBERTa model.
    tokenizer:          Matching tokenizer.
    retrieval:          RetrievalPipeline instance.
    config:             Global PipelineConfig.
    contradiction_only: If True, only flag CONTRADICTION as hallucination.
                        If False, also flag NEUTRAL (unverifiable claims).
    confidence_threshold: Minimum confidence to trust a prediction.
                          Below this, hallucination_flag is True regardless.
    """

    # Sentence splitter: split on .  !  ?  followed by whitespace + capital
    _SENT_RE = re.compile(
        r'(?<=[.!?])\s+(?=[A-Z])'  # standard sentence boundary
        r'|(?<=\d)\. (?=[A-Z])',    # number-terminated sentence
    )
    _MIN_CLAIM_WORDS = 5   # discard trivially short fragments

    def __init__(
        self,
        model,
        tokenizer,
        retrieval: RetrievalPipeline,
        config: PipelineConfig,
        contradiction_only: bool = False,
        confidence_threshold: float = 0.50,
    ) -> None:
        self.model      = model
        self.tokenizer  = tokenizer
        self.retrieval  = retrieval
        self.cfg        = config
        self.flag_contradiction_only = contradiction_only
        self.confidence_threshold    = confidence_threshold
        self.model.eval() 

    def detect(
        self,
        summary: str,
        ticker_filter:  Optional[str] = None,
        date_after:     Optional[str] = None,
        date_before:    Optional[str] = None,
        top_k_evidence: Optional[int] = None,
    ) -> List[ClaimVerdict]:
        """Detect hallucinations in a financial LLM summary.

        Parameters
        ----------
        summary:        Raw LLM-generated text to verify.
        ticker_filter:  Restrict evidence to a specific company.
        date_after:     Earliest filing date to consider (YYYY-MM-DD).
        date_before:    Latest filing date to consider (YYYY-MM-DD).
        top_k_evidence: Evidence chunks retrieved per claim.

        Returns
        -------
        List of ClaimVerdict, one per claim detected in *summary*.
        """
        top_k = top_k_evidence or self.cfg.TOP_K
        claims = self._split_claims(summary)
        log.info('Detected %d claims in summary', len(claims))

        verdicts: List[ClaimVerdict] = []
        for idx, claim in enumerate(tqdm(claims, desc='Verifying claims')):
            evidence_list = self.retrieval.retrieve(
                claim=claim,
                top_k=top_k,
                ticker_filter=ticker_filter,
                date_after=date_after,
                date_before=date_before,
            )

            if not evidence_list:
                # No evidence found → unverifiable → flag
                verdicts.append(ClaimVerdict(
                    claim_index=idx, claim=claim,
                    evidence_text='(no evidence retrieved)',
                    evidence_ticker='', evidence_section='',
                    evidence_date='', evidence_page=0,
                    prediction='NEUTRAL',
                    confidence=0.0,
                    prob_entailment=0.0, prob_neutral=1.0, prob_contradiction=0.0,
                    hallucination_flag=True,
                    retrieval_score=0.0,
                ))
                continue

            # Use the highest-scoring evidence chunk for NLI
            top_evidence = evidence_list[0]
            probs = self._nli_predict(top_evidence.text, claim)
            pred_id    = int(np.argmax(probs))
            # Use model.config.id2label as the single source of truth.
            # Fix 4 normalises these to uppercase strings so downstream
            # flag logic (which checks label == 'CONTRADICTION') is safe.
            pred_label = self.model.config.id2label[pred_id]
            confidence = float(probs[pred_id])

            flag = self._should_flag(pred_label, confidence)

            verdicts.append(ClaimVerdict(
                claim_index=idx,
                claim=claim,
                evidence_text=top_evidence.text,
                evidence_ticker=top_evidence.ticker,
                evidence_section=top_evidence.section_title,
                evidence_date=top_evidence.filing_date,
                evidence_page=top_evidence.page,
                prediction=pred_label,
                confidence=confidence,
                # Index into probs using the model's own label2id so that
                # the per-class probabilities are always correctly assigned
                # regardless of which base model or checkpoint is loaded.
                prob_entailment=float(probs[
                    self.model.config.label2id.get('ENTAILMENT', LABEL2ID['ENTAILMENT'])]),
                prob_neutral=float(probs[
                    self.model.config.label2id.get('NEUTRAL', LABEL2ID['NEUTRAL'])]),
                prob_contradiction=float(probs[
                    self.model.config.label2id.get('CONTRADICTION', LABEL2ID['CONTRADICTION'])]),
                hallucination_flag=flag,
                retrieval_score=top_evidence.score,
            ))

        self._save_report(summary, verdicts)
        return verdicts

    def summary_stats(self, verdicts: List[ClaimVerdict]) -> Dict[str, Any]:
        """Return aggregate statistics over a list of ClaimVerdicts."""
        if not verdicts:
            return {}
        n = len(verdicts)
        flagged = sum(1 for v in verdicts if v.hallucination_flag)
        label_counts = {}
        for v in verdicts:
            label_counts[v.prediction] = label_counts.get(v.prediction, 0) + 1

        return {
            'total_claims':          n,
            'flagged_claims':        flagged,
            'hallucination_rate':    round(flagged / n, 4),
            'label_distribution':    label_counts,
            'mean_confidence':       round(float(np.mean([v.confidence for v in verdicts])), 4),
            'mean_retrieval_score':  round(float(np.mean([v.retrieval_score for v in verdicts])), 4),
        }

    # claim splitting

    def _split_claims(self, text: str) -> List[str]:
        """Split a financial summary into individual verifiable claims.

        Uses sentence-boundary detection; short fragments are merged with
        the preceding claim to avoid trivial NLI inputs.
        """
        raw_sentences = self._SENT_RE.split(text.strip())
        claims: List[str] = []
        buffer = ''

        for sent in raw_sentences:
            sent = sent.strip()
            if not sent:
                continue
            word_count = len(sent.split())
            if word_count < self._MIN_CLAIM_WORDS:
                buffer = (buffer + ' ' + sent).strip()
                continue
            if buffer:
                claims.append(buffer)
                buffer = ''
            claims.append(sent)

        if buffer:
            if claims:
                claims[-1] = (claims[-1] + ' ' + buffer).strip()
            else:
                claims.append(buffer)

        return claims

    # NLI inference 

    @torch.no_grad()
    def _nli_predict(self, premise: str, hypothesis: str) -> np.ndarray:
        """Run NLI inference on a single (premise, hypothesis) pair.

        Returns a (3,) softmax probability array.
        """
        enc = self.tokenizer(
            premise, hypothesis,
            truncation=True,
            padding=True,
            max_length=self.cfg.MAX_SEQ_LENGTH,
            return_tensors='pt',
        ).to(DEVICE)

        logits = self.model(**enc).logits[0]        # (num_labels,)
        return torch.softmax(logits, dim=-1).cpu().numpy()

    # flagging logic 

    def _should_flag(self, label: str, confidence: float) -> bool:
        """Determine whether to raise a hallucination flag.

        Flagging rules (configurable):
        - Always flag CONTRADICTION.
        - Flag NEUTRAL if contradiction_only=False (claim unverifiable).
        - Flag any prediction below the confidence threshold.
        """
        if label == 'CONTRADICTION':
            return True
        if not self.flag_contradiction_only and label == 'NEUTRAL':
            return True
        if confidence < self.confidence_threshold:
            return True
        return False

    # output 

    def _save_report(self, summary: str, verdicts: List[ClaimVerdict]) -> None:
        report = {
            'summary_preview': summary[:200],
            'stats':           self.summary_stats(verdicts),
            'verdicts':        [v.to_dict() for v in verdicts],
        }
        out = self.cfg.OUTPUTS_DIR / 'inference_report.json'
        out.write_text(json.dumps(report, indent=2, default=str), encoding='utf-8')
        log.info('Inference report saved → %s', out)


log.info('HallucinationDetector class defined')


2026-07-02 19:04:36 | INFO     | hallucination_engine | HallucinationDetector class defined


In [93]:
# Instantiate and run inference smoke-test
detector = HallucinationDetector(
    model=nli_model,
    tokenizer=nli_tokenizer,
    retrieval=retrieval_pipeline,
    config=cfg,
    contradiction_only=False,   # flag both CONTRADICTION and NEUTRAL
    confidence_threshold=0.50,
)

DEMO_SUMMARY = (
    "Apple's fiscal year 2023 results showed strong performance across the board. "
    "Total net sales reached $383.3 billion, representing a 2.8 percent decline compared to fiscal 2022. "
    "iPhone revenue was $200.6 billion, accounting for 52 percent of total net sales. "
    "Services revenue surged to $100 billion, setting a new company record. "
    "Microsoft Cloud revenue grew 22 percent to $111.6 billion in its fiscal 2023. "
    "Azure grew just 10 percent year-over-year, below analyst expectations."
)

verdicts = detector.detect(
    summary=DEMO_SUMMARY,
    ticker_filter=None,   # search all tickers
)

stats = detector.summary_stats(verdicts)
print(f'\nSummary stats: {stats}\n')

2026-07-02 19:04:36 | INFO     | hallucination_engine | Detected 5 claims in summary


Verifying claims:   0%|          | 0/5 [00:00<?, ?it/s]

2026-07-02 19:04:36 | INFO     | hallucination_engine | Inference report saved → /home/lonewolf/quant/Projects/FINANCIAL_LLM/v3/outputs/inference_report.json

Summary stats: {'total_claims': 5, 'flagged_claims': 5, 'hallucination_rate': 1.0, 'label_distribution': {'CONTRADICTION': 1, 'NEUTRAL': 4}, 'mean_confidence': 0.8605, 'mean_retrieval_score': 0.5824}



In [94]:
# Pretty-print verdict table 
ICONS = {'ENTAILMENT': '✅', 'NEUTRAL': '⚪', 'CONTRADICTION': '❌'}

print(f"{'#':<3} {'PRED':<15} {'CONF':>5}  {'FLAG':>5}  CLAIM")
print('-' * 80)
for v in verdicts:
    icon   = ICONS.get(v.prediction, '?')
    flag   = '🚨' if v.hallucination_flag else '  '
    claim_ = v.claim[:70] + ('…' if len(v.claim) > 70 else '')
    print(f"{v.claim_index:<3} {icon} {v.prediction:<13} {v.confidence:>5.3f}  {flag}  {claim_}")

print()
print('Evidence snippets for flagged claims:')
for v in verdicts:
    if v.hallucination_flag:
        print(f'\n  Claim: {v.claim[:100]}')
        print(f'  Prediction: {v.prediction} (conf={v.confidence:.3f})')
        print(f'  Evidence [{v.evidence_ticker}/{v.evidence_date}]: {v.evidence_text[:200]}...')


#   PRED             CONF   FLAG  CLAIM
--------------------------------------------------------------------------------
0   ❌ CONTRADICTION 0.549  🚨  Apple's fiscal year 2023 results showed strong performance across the …
1   ⚪ NEUTRAL       0.995  🚨  Total net sales reached $383.3 billion, representing a 2.8 percent dec…
2   ⚪ NEUTRAL       0.976  🚨  Services revenue surged to $100 billion, setting a new company record.
3   ⚪ NEUTRAL       0.905  🚨  Microsoft Cloud revenue grew 22 percent to $111.6 billion in its fisca…
4   ⚪ NEUTRAL       0.877  🚨  Azure grew just 10 percent year-over-year, below analyst expectations.

Evidence snippets for flagged claims:

  Claim: Apple's fiscal year 2023 results showed strong performance across the board.
  Prediction: CONTRADICTION (conf=0.549)
  Evidence [UNKNOWN/2026-07-02]: exhibit number exhibit description form exhibit 4. 10 officer ’ s certificate of the registrant, dated as of february 23, 2016, including forms of global notes representin

# END to END

In [95]:
from transformers import AutoTokenizer as _AutoTok
from sentence_transformers import SentenceTransformer as _STModel


class EndToEndPipeline:
    """Orchestrator that wires together every module from Sections 3–13.

    Designed for programmatic use and as the foundation of a REST API.

    Parameters
    ----------
    config:    Global PipelineConfig.
    detector:  Pre-built HallucinationDetector (Section 13).
    """

    def __init__(
        self,
        config: PipelineConfig,
        detector: HallucinationDetector,
    ) -> None:
        self.cfg      = config
        self.detector = detector


    def run(
        self,
        summary: str,
        ticker: Optional[str] = None,
        date_after:  Optional[str] = None,
        date_before: Optional[str] = None,
        top_k: int = 3,
    ) -> Dict[str, Any]:
        """Run end-to-end hallucination detection on *summary*.

        Parameters
        ----------
        summary:     LLM-generated financial text to verify.
        ticker:      Optional ticker filter for evidence retrieval.
        date_after:  Evidence filing date lower bound.
        date_before: Evidence filing date upper bound.
        top_k:       Evidence chunks retrieved per claim.

        Returns
        -------
        Dict with 'verdicts', 'stats', 'flagged', 'html_report'.
        """
        log.info('E2E pipeline invoked | ticker=%s | top_k=%d', ticker, top_k)
        t0 = time.time()

        verdicts = self.detector.detect(
            summary=summary,
            ticker_filter=ticker,
            date_after=date_after,
            date_before=date_before,
            top_k_evidence=top_k,
        )

        stats    = self.detector.summary_stats(verdicts)
        flagged  = [v for v in verdicts if v.hallucination_flag]
        elapsed  = time.time() - t0

        result = {
            'verdicts':        [v.to_dict() for v in verdicts],
            'stats':           stats,
            'flagged_indices': [v.claim_index for v in flagged],
            'elapsed_seconds': round(elapsed, 2),
            'html_report':     self._render_html(summary, verdicts, stats),
        }

        # Persist
        out_json = self.cfg.OUTPUTS_DIR / 'e2e_demo_report.json'
        out_txt  = self.cfg.OUTPUTS_DIR / 'e2e_demo_summary.txt'
        out_json.write_text(
            json.dumps({k: v for k, v in result.items() if k != 'html_report'},
                        indent=2, default=str),
            encoding='utf-8')
        out_txt.write_text(self._render_text(summary, verdicts, stats), encoding='utf-8')

        log.info(
            'E2E complete in %.1fs — %d claims, %d flagged (rate=%.1f%%)',
            elapsed, stats.get('total_claims', 0),
            stats.get('flagged_claims', 0),
            stats.get('hallucination_rate', 0) * 100,
        )
        return result

    # output renderers 

    @staticmethod
    def _render_text(
        summary: str,
        verdicts: List[ClaimVerdict],
        stats: Dict[str, Any],
    ) -> str:
        """Plain-text report suitable for logging or email."""
        lines = [
            '=' * 72,
            'HALLUCINATION DETECTION REPORT',
            'Thomson Reuters · Trustworthy AI',
            f'Generated: {datetime.utcnow().isoformat()}Z',
            '=' * 72,
            '',
            'ORIGINAL SUMMARY:',
            summary,
            '',
            'AGGREGATE STATISTICS:',
        ]
        for k, v in stats.items():
            lines.append(f'  {k:<28}: {v}')

        lines += ['', 'CLAIM-LEVEL VERDICTS:', '-' * 72]
        icons = {'ENTAILMENT': '[OK]', 'NEUTRAL': '[??]', 'CONTRADICTION': '[!!]'}
        for v in verdicts:
            icon = icons.get(v.prediction, '[?]')
            flag = 'HALLUCINATION' if v.hallucination_flag else '          OK'
            lines += [
                f'',
                f'#{v.claim_index+1:02d} {icon} {flag}  conf={v.confidence:.3f}',
                f'    CLAIM    : {v.claim}',
                f'    EVIDENCE : [{v.evidence_ticker}/{v.evidence_date}] {v.evidence_text[:180]}',
                f'    P(E/N/C) : {v.prob_entailment:.3f} / {v.prob_neutral:.3f} / {v.prob_contradiction:.3f}',
            ]

        lines += ['', '=' * 72]
        return '\n'.join(lines)

    @staticmethod
    def _render_html(
        summary: str,
        verdicts: List[ClaimVerdict],
        stats: Dict[str, Any],
    ) -> str:
        """Minimal self-contained HTML report for browser or email rendering."""
        colour_map = {
            'ENTAILMENT':    '#d4edda',  # green tint
            'NEUTRAL':       '#fff3cd',  # amber tint
            'CONTRADICTION': '#f8d7da',  # red tint
        }

        rows = ''
        for v in verdicts:
            bg = colour_map.get(v.prediction, '#fff')
            flag_badge = ('<span style="color:red;font-weight:bold">🚨 FLAGGED</span>'
                          if v.hallucination_flag else
                          '<span style="color:green">✅ OK</span>')
            rows += f"""
            <tr style="background:{bg}">
              <td style="padding:6px;font-weight:bold">{v.claim_index+1}</td>
              <td style="padding:6px">{v.claim}</td>
              <td style="padding:6px">{v.prediction}</td>
              <td style="padding:6px">{v.confidence:.3f}</td>
              <td style="padding:6px">{flag_badge}</td>
              <td style="padding:6px;font-size:0.85em;color:#555">{v.evidence_text[:120]}…</td>
            </tr>"""

        stat_rows = ''.join(
            f'<tr><td><b>{k}</b></td><td>{v}</td></tr>'
            for k, v in stats.items()
        )

        return f"""<!DOCTYPE html>
<html><head><meta charset="utf-8">
<title>Hallucination Detection Report</title>
<style>body{{font-family:sans-serif;padding:20px}}table{{border-collapse:collapse;width:100%}}
th{{background:#333;color:#fff;padding:8px}}td{{border:1px solid #ddd}}</style></head>
<body>
<h1>🔍 Hallucination Detection Report</h1>
<p><em>Thomson Reuters · Trustworthy AI · {datetime.utcnow().strftime('%Y-%m-%d %H:%M')} UTC</em></p>
<h2>Summary Statistics</h2>
<table width="40%"><tr><th>Metric</th><th>Value</th></tr>{stat_rows}</table>
<h2>Claim Verdicts</h2>
<table><tr><th>#</th><th>Claim</th><th>Prediction</th><th>Confidence</th>
<th>Flag</th><th>Evidence</th></tr>{rows}</table>
</body></html>"""


log.info('EndToEndPipeline class defined')


2026-07-02 19:04:37 | INFO     | hallucination_engine | EndToEndPipeline class defined


In [96]:
DEMO_LLM_SUMMARY = """
Apple Inc. reported fiscal year 2023 net sales of $383.3 billion, which represented
a decrease of 2.8 percent compared to fiscal year 2022. iPhone revenue totalled
$200.6 billion and accounted for 52 percent of total revenue. Services revenue
hit a new record of $95 billion, growing 15 percent year-over-year. Apple's
management highlighted significant competition risks, noting that the smartphone
market remains highly competitive. Microsoft reported fiscal 2023 revenue of
$211.9 billion, growing 7 percent. Azure cloud services grew 29 percent
year-over-year while Microsoft's operating income was $88.5 billion.
Microsoft net income for the year was $50 billion.
""".strip()

DEMO_TICKER: Optional[str] = None  # None = search all tickers

e2e = EndToEndPipeline(config=cfg, detector=detector)
demo_result = e2e.run(
    summary=DEMO_LLM_SUMMARY,
    ticker=DEMO_TICKER,
    top_k=3,
)

print(demo_result['stats'])


2026-07-02 19:04:37 | INFO     | hallucination_engine | E2E pipeline invoked | ticker=None | top_k=3
2026-07-02 19:04:37 | INFO     | hallucination_engine | Detected 6 claims in summary


Verifying claims:   0%|          | 0/6 [00:00<?, ?it/s]

2026-07-02 19:04:37 | INFO     | hallucination_engine | Inference report saved → /home/lonewolf/quant/Projects/FINANCIAL_LLM/v3/outputs/inference_report.json
2026-07-02 19:04:37 | INFO     | hallucination_engine | E2E complete in 0.2s — 6 claims, 6 flagged (rate=100.0%)
{'total_claims': 6, 'flagged_claims': 6, 'hallucination_rate': 1.0, 'label_distribution': {'NEUTRAL': 6}, 'mean_confidence': 0.9836, 'mean_retrieval_score': 0.6015}


In [97]:
txt_path = cfg.OUTPUTS_DIR / 'e2e_demo_summary.txt'
print(txt_path.read_text(encoding='utf-8'))

HALLUCINATION DETECTION REPORT
Thomson Reuters · Trustworthy AI
Generated: 2026-07-02T19:04:37.421624Z

ORIGINAL SUMMARY:
Apple Inc. reported fiscal year 2023 net sales of $383.3 billion, which represented
a decrease of 2.8 percent compared to fiscal year 2022. iPhone revenue totalled
$200.6 billion and accounted for 52 percent of total revenue. Services revenue
hit a new record of $95 billion, growing 15 percent year-over-year. Apple's
management highlighted significant competition risks, noting that the smartphone
market remains highly competitive. Microsoft reported fiscal 2023 revenue of
$211.9 billion, growing 7 percent. Azure cloud services grew 29 percent
year-over-year while Microsoft's operating income was $88.5 billion.
Microsoft net income for the year was $50 billion.

AGGREGATE STATISTICS:
  total_claims                : 6
  flagged_claims              : 6
  hallucination_rate          : 1.0
  label_distribution          : {'NEUTRAL': 6}
  mean_confidence             : 0.9

In [98]:
from IPython.display import HTML, display

html_out = cfg.OUTPUTS_DIR / 'e2e_demo_report.html'
html_out.write_text(demo_result['html_report'], encoding='utf-8')
log.info('HTML report saved → %s', html_out)

display(HTML(demo_result['html_report']))


2026-07-02 19:04:37 | INFO     | hallucination_engine | HTML report saved → /home/lonewolf/quant/Projects/FINANCIAL_LLM/v3/outputs/e2e_demo_report.html


In [99]:
def _list_outputs(base: Path, prefix: str = '') -> None:
    for item in sorted(base.iterdir()):
        size = item.stat().st_size if item.is_file() else 0
        tag  = f'({size/1024:.1f} KB)' if item.is_file() else ''
        print(f'{prefix}{item.name} {tag}')
        if item.is_dir() and item.name not in {'chroma', '__pycache__'}:
            _list_outputs(item, prefix + '  ')


print('\n── Project output artefacts ──')
_list_outputs(cfg.PROJECT_ROOT / 'data')
_list_outputs(cfg.PROJECT_ROOT / 'models')
_list_outputs(cfg.OUTPUTS_DIR)

print('\n── Pipeline complete ──')
print('All artefacts are persisted and reproducible from PipelineConfig.')


── Project output artefacts ──
annotations 
  annotations.db (8.0 KB)
  annotations_export.parquet (9.9 KB)
  test.csv (0.5 KB)
  train.csv (1.4 KB)
  validation.csv (0.3 KB)
chunks 
  chunks.parquet (959.5 KB)
embeddings 
  embeddings.npy (3726.1 KB)
  metadata.paarquet (25.5 KB)
parsed 
  UNKNOWN_UNKNOWN_2026_07_02.json (274.8 KB)
  parsed_manifest.parquet (5.7 KB)
pdf 
  AAPL_10K.pdf:Zone.Identifier (0.0 KB)
  MSFT_10K.pdf:Zone.Identifier (0.0 KB)
raw_document_manifest.parquet (7.1 KB)
raw_documents 
  AAPL_10K.pdf (2561.9 KB)
  GOOG_10K_2025.pdf (4055.0 KB)
  GOOG_10K_2025.pdf:Zone.Identifier (0.0 KB)
  GOOG_10K_2026.pdf (4115.8 KB)
  GOOG_10K_2026.pdf:Zone.Identifier (0.0 KB)
  MSFT_10K.pdf (8461.7 KB)
  NVDA_10K_2025.pdf (3128.8 KB)
  NVDA_10K_2025.pdf:Zone.Identifier (0.0 KB)
  NVDA_10K_2026.pdf (2944.6 KB)
  NVDA_10K_2026.pdf:Zone.Identifier (0.0 KB)
  TSLA_10K_2025.pdf (2838.7 KB)
  TSLA_10K_2025.pdf:Zone.Identifier (0.0 KB)
  TSLA_10K_2026.pdf (1902.2 KB)
  TSLA_10K_2026.pdf